# 한국 배당성장주 통합 스크리너 · 매년 4~5월 실전 리밸런싱 (Google Colab)

코스피·코스닥 전체 상장사를 대상으로 **현행 배당성장 규칙(1~9번, 전략 A)**을 스캔하고,
본인 비공개 **Google 스프레드시트(또는 Drive YAML)**와 실시간 연동해 **전체 바구니 비중 · 3대 상한선 · 보유 한국주 정기 검진 · 4~5월 매수 배분표**까지 1클릭 A4 PDF로 정리하는 실전 노트북입니다.

### 개인정보 보호 설계 (Public 저장소 안전 원칙)
- **GitHub에는 개인정보 0%**: 스프레드시트 ID, 계좌명, 보유 종목, 평가금액은 코드에 전혀 저장되지 않습니다.
- **실행 시 본인 Google 계정으로만 읽기**: Colab 좌측 열쇠 아이콘(**Secrets**)에 `DART_API_KEY`와 `PORTFOLIO_SHEET_ID`를 등록해 두면, 실행 시 본인 권한으로만 시트를 읽어오며 **계좌명·소유자 이름은 집계 단계에서 즉시 제외**되어 리포트에 절대 출력되지 않습니다.
- **금액 비공개 옵션**: 리포트 화면을 캡처·공유할 일이 있다면 `SCREENER_CONFIG["MASK_AMOUNTS_IN_REPORT"] = True`로 켜면 평가금액을 숨기고 **비중(%)만** 표시합니다.

### 매년 4~5월 실행 방법
1. Colab 좌측 **Secrets(열쇠 아이콘)**에 아래 2개를 등록하고 `노트북 액세스` ON:
   - `DART_API_KEY`: OpenDART 무료 API 키
   - `PORTFOLIO_SHEET_ID`: 본인 포트폴리오 Google 스프레드시트 ID *(URL의 `/d/`와 `/edit` 사이 문자열, 선택 사항)*
2. 상단 메뉴 **`런타임 → 모두 실행`** 클릭 (Google 인증 팝업이 뜨면 본인 계정 허용)
3. 완료 후 Google Drive(`Colab Notebooks/dividend_screener/reports/`)에 날짜·시각별 PDF가 자동 누적 저장됩니다.


In [ ]:
# ==============================================================================
# 1. 환경 설정, 한글 폰트 자동 구성 및 OpenDART API 키 안전 로드
# ==============================================================================
import os, sys, re, io, gzip, json, math, time, base64, zipfile, getpass, warnings, statistics
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import date, datetime, timezone, timedelta
from html import escape
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트(fonts-nanum) 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "libpango-1.0-0", "libpangoft2-1.0-0"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "weasyprint", "pyyaml", "gspread"], check=False)

import yaml
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mtick

try:
    from IPython.display import display
except ImportError:
    display = print

def setup_korean_font():
    chosen = None
    for p in [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    ]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            if chosen is None:
                chosen = fm.FontProperties(fname=p).get_name()
    if chosen is None:
        avail = {f.name for f in fm.fontManager.ttflist}
        for cand in ["NanumGothic", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
            if cand in avail:
                chosen = cand
                break
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass
    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 120
    return chosen

def load_dart_api_key() -> str:
    key = ""
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = (userdata.get("DART_API_KEY") or "").strip()
        except Exception:
            key = ""
    if not key:
        key = os.environ.get("DART_API_KEY", "").strip()
    if not key:
        key = getpass.getpass("OpenDART API Key(DART_API_KEY)를 입력하세요 (화면 미표시): ").strip()
    if not key:
        raise RuntimeError("DART_API_KEY가 필요합니다. OpenDART(opendart.fss.or.kr)에서 무료 키를 발급받아 입력해 주세요.")
    return key

FONT_NAME = setup_korean_font()
DART_API_KEY = load_dart_api_key()
print(f"[완료] 한글 폰트: {FONT_NAME} | DART_API_KEY 로드 완료 (길이 {len(DART_API_KEY)}자, 마스킹 보호)")


In [ ]:
# ==============================================================================
# 2. 실행 설정 · Google Drive 연결 · 비공개 Google 스프레드시트 / YAML 포트폴리오 로드
# ==============================================================================
SCREENER_CONFIG = {
    # 기준 사업연도 (매년 4~5월에는 직전 연도, 예: 2027년 4월 실행 시 2026)
    "FISCAL_YEAR": 2025,

    # 탐색 대상 시장 및 시총 하한 (단위: 억 원)
    # - 3000: 시총 3,000억 이상 (~600종목, 최초 약 3~5분 / 이어받기 30~60초)
    # - 1000: 시총 1,000억 이상 (~1,200종목, 최초 약 6~8분)
    # -    0: 코스피·코스닥 전 종목 (~2,500종목, 최초 약 12~18분)
    "MARKETS": ["KOSPI", "KOSDAQ"],
    "MIN_MCAP_EOK": 3000,

    # 1차 배당 이력 선별 기준
    "MIN_CONSECUTIVE_DIV_YEARS": 10,  # 최근 10년 연속 배당 (원형 1번 PASS 기준)
    "MAX_CUTS_5Y": 2,                 # 최근 5년 DPS 삭감 2회 이하
    "MAX_PAYOUT_3Y": 75.0,            # 3년 합 배당성향 75% 이하

    # 실전 리밸런싱 유동성 기준 (7.7년 백테스트에서 유일하게 성과 개선 검증 -> 배지 표시 & 신규매수 우선)
    "PREFER_LIQUIDITY_EOK": 10.0,

    # 리포트 금액 마스킹 옵션 (True로 켜면 PDF/HTML에서 평가금액 숫자를 숨기고 비중%만 표시)
    "MASK_AMOUNTS_IN_REPORT": False,

    # Google 스프레드시트 연동 설정 (시트 ID는 Colab Secrets 'PORTFOLIO_SHEET_ID' 또는 비공개 Drive YAML에서 로드)
    "USE_GOOGLE_SHEET": True,
    "SHEET_TAB_NAME": "보유종목",      # 계좌별 보유 종목이 들어 있는 탭 이름

    # 보완 필터 강제 제외 토글 (기본 False = 현행 전략 A 유지, 원할 때만 True)
    "USE_YIELD_GUARD": False,
    "MIN_YIELD": 0.02,
    "MAX_YIELD": 0.12,
    "USE_DPS_CAGR_FLOOR": False,
    "MIN_DPS_CAGR_5Y": 0.03,
    "USE_LIQUIDITY_GUARD": False,
    "MIN_TURNOVER_60D": 1_000_000_000,
    "USE_CRASH_EPS_GUARD": False,
    "MAX_DD_52W_WITH_EPS_DROP": -0.35,
    "USE_VAL_CAP": False,
    "VAL_CAP": 0.50,

    # Google Drive 이어받기 캐시 · 비공개 포트폴리오 YAML · 리포트 저장 경로
    "USE_GOOGLE_DRIVE": True,
    "DRIVE_DIR": "/content/drive/MyDrive/Colab Notebooks/dividend_screener",
    "PORTFOLIO_FILENAME": "my_kr_portfolio.yaml",
    "FORCE_REFRESH_CACHE": False,

    # 병렬 작업자 수 및 로컬 임시 캐시 경로
    "WORKERS": 8,
    "CACHE_DIR": "./_cache_screener",

    # 국고채 10년물 연평균 금리 CSV 소스 (Common_Everything Public)
    "GITHUB_REPO": "khs0221/Common_Everything",
    "GITHUB_BRANCHES": ["feat/dividend-backtest", "main", "copilot/ai-coding-curriculum"],
    "DATA_SUBPATH": "dividend-backtest/data",
}

# (선택) Drive YAML 대신 임시 테스트용 딕셔너리 (공개 GitHub 커밋 시 반드시 None 유지!)
INLINE_PORTFOLIO = None

PORTFOLIO_YAML_TEMPLATE = """# ==============================================================================
# 내 포트폴리오 설정 (본인 비공개 Google Drive 전용 - 공개 GitHub 미업로드)
# 위치: MyDrive/Colab Notebooks/dividend_screener/my_kr_portfolio.yaml
# ==============================================================================
# [가장 편한 사용법: Google 스프레드시트 자동 연동]
#   Colab 좌측 열쇠(Secrets)에 PORTFOLIO_SHEET_ID를 등록하거나, 아래 sheet_id에 붙여넣으세요.
#   (계좌명·가족 이름은 집계 시 즉시 제외되어 PDF/HTML 리포트에 절대 표시되지 않습니다!)
# ------------------------------------------------------------------------------

sheet_id: ""                    # (선택) Colab Secrets 미사용 시 여기에 스프레드시트 ID 입력
sheet_tab: "보유종목"            # 보유 종목 탭 이름

# 전체 포트폴리오 목표 바구니 비중 (목표, 하한, 상한)
basket_targets:
  A: {name: "국내 배당성장", target: 0.50, low: 0.45, high: 0.55}
  B: {name: "반도체·AI",     target: 0.20, low: 0.15, high: 0.25}
  C: {name: "미국 지수",     target: 0.30, low: 0.25, high: 0.35}

# 전체 투자자산 기준 3대 상한선 (운용규칙 3절)
single_stock_cap: 0.15          # 개별 회사 주식 단일 상한 15% (ETF 제외)
financial_cap: 0.35             # 국내 금융 업종 합계 상한 35%
ai_cap: 0.30                    # AI·반도체 실제 노출 상한 30%

# 이번 4~5월 리밸런싱 때 투입할 신규 자금 + 누적 배당금 (원 단위, 0이면 매수 배분표 생략)
new_cash_krw: 0                 # 예: 10000000 (1,000만 원)
include_sell_proceeds: false    # true: '매도 검토' 종목 평가액도 매수 재원에 합산
max_new_picks: 4                # 신규 자금을 나눠 담을 최대 종목 수
prefer_liquidity_eok: 10.0      # 신규 매수 시 60일 일평균 거래대금 최소 기준 (억 원)

# (선택) 스프레드시트 종목별 바구니·트랙·AI노출 세밀 지정 (안 적어도 종목명 기준으로 자동 분류됨)
# - track: dividend(기본) | total_return(자사주소각 총환원형) | cycle(경기 사이클 별도 관리)
# - basket: A(국내 배당성장) | B(반도체·AI) | C(미국 지수)
# - ai: AI·반도체 노출 계수 (0.0 ~ 1.0)
overrides: {}
  # "005935": {basket: "B", track: "cycle", ai: 1.0, common_ticker: "005930"}

# (선택) 시총 3,000억 미만이더라도 꼭 함께 정밀 점검할 관심 종목 티커
watchlist: []
  # - "005380"

# (선택) 스프레드시트를 쓰지 않을 때만 수동으로 보유 종목을 적으세요 (시트 연동 시 자동 무시됨)
kr_basket_ratio_of_total: 0.50
holdings: []
"""

CACHE_DIR = Path(SCREENER_CONFIG["CACHE_DIR"])
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_ROOT = None
DRIVE_REPORT_DIR = None
PORTFOLIO_PATH = None
if IN_COLAB and SCREENER_CONFIG["USE_GOOGLE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(SCREENER_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        PORTFOLIO_PATH = DRIVE_ROOT / SCREENER_CONFIG["PORTFOLIO_FILENAME"]
        print(f"[Drive 연결 완료] 경로: {DRIVE_ROOT}")
    except Exception as e:
        print(f"[Drive 미연결] 로컬 임시 저장소로 진행합니다 ({type(e).__name__}: {e})")

def save_checkpoint(filename: str, data: dict) -> None:
    """단일 압축 파일로 빠르게 체크포인트 저장 (Drive FUSE 부하 최소화)."""
    local_p = CACHE_DIR / filename
    raw = gzip.compress(json.dumps(data, ensure_ascii=False, separators=(",", ":")).encode("utf-8"), compresslevel=4)
    local_p.write_bytes(raw)
    if DRIVE_ROOT is not None:
        try:
            tmp_p = DRIVE_ROOT / (filename + ".tmp")
            dst_p = DRIVE_ROOT / filename
            tmp_p.write_bytes(raw)
            tmp_p.replace(dst_p)
        except Exception as e:
            print(f"  [주의] Drive 체크포인트 저장 지연({type(e).__name__}) — 로컬에는 보존됨")

def load_checkpoint(filename: str) -> dict:
    if SCREENER_CONFIG["FORCE_REFRESH_CACHE"]:
        return {}
    for base in ([DRIVE_ROOT, CACHE_DIR] if DRIVE_ROOT is not None else [CACHE_DIR]):
        p = base / filename
        if p.exists() and p.stat().st_size > 0:
            try:
                return json.loads(gzip.decompress(p.read_bytes()).decode("utf-8"))
            except Exception:
                continue
    return {}

_ETF_NAME_RE = re.compile(r"(TIGER|KODEX|ACE|SOL|KBSTAR|PLUS|RISE|ARIRANG|HANARO|TIMEFOLIO|KOSEF|ETF|ETN)", re.I)
_AI_BASKET_RE = re.compile(r"(반도체|필라델피아|AI|인공지능|소부장)", re.I)
_US_BASKET_RE = re.compile(r"(미국|S&P|나스닥|NASDAQ|US\s|다우존스|SCHD|QQQ|SPY|VOO|Tech)", re.I)

def _parse_sheet_num(v) -> float:
    if v in (None, "", "-", "－"):
        return 0.0
    s = str(v).replace(",", "").replace("원", "").replace("₩", "").replace("$", "").replace("%", "").strip()
    try:
        x = float(s)
        return 0.0 if (math.isnan(x) or math.isinf(x)) else x
    except ValueError:
        return 0.0

def _extract_sheet_id(raw: str) -> str:
    s = str(raw or "").strip()
    m = re.search(r"/spreadsheets/d/([A-Za-z0-9_-]+)", s)
    return m.group(1) if m else s

def _classify_position(ticker: str, name: str, exchange: str, ov: dict) -> dict:
    """공개 가능한 일반 규칙으로 바구니(A/B/C)·ETF 여부·AI 노출 계수 자동 추정 (개인 설정 overrides 우선)."""
    nm = str(name or "")
    is_foreign = exchange.upper() not in ("", "KRX", "KOSPI", "KOSDAQ")
    is_etf = bool(ov.get("etf")) if "etf" in ov else (is_foreign or bool(_ETF_NAME_RE.search(nm)))

    if "basket" in ov:
        basket = str(ov["basket"]).upper()
    elif is_etf:
        if _AI_BASKET_RE.search(nm):
            basket = "B"
        else:
            basket = "C"
    else:
        basket = "A"

    if "ai" in ov:
        ai_coef = float(ov["ai"])
    elif basket == "B":
        ai_coef = 1.0
    elif basket == "C":
        if re.search(r"나스닥|NASDAQ|QQQ", nm, re.I):
            ai_coef = 0.50
        elif re.search(r"Tech|빅테크|Top10", nm, re.I):
            ai_coef = 0.40
        elif re.search(r"S&P|SPY|VOO", nm, re.I):
            ai_coef = 0.35
        elif re.search(r"배당다우|SCHD|배당", nm, re.I):
            ai_coef = 0.05
        else:
            ai_coef = 0.25 if _US_BASKET_RE.search(nm) else 0.0
    else:
        ai_coef = 0.0

    track = str(ov.get("track") or ("cycle" if (not is_etf and basket == "B") else "dividend")).strip().lower()
    return {"is_etf": is_etf, "is_foreign": is_foreign, "basket": basket, "ai": ai_coef, "track": track}

def parse_sheet_rows_aggregated(rows: list[list], overrides: dict) -> list[dict]:
    """스프레드시트 '보유종목' 탭 행 목록을 종목별로 합산 (계좌명·소유자명은 즉시 폐기!)."""
    if not rows or len(rows) < 2:
        return []
    header = [str(h or "").strip() for h in rows[0]]
    def find_col(pats, fallback_idx):
        for idx, h in enumerate(header):
            if any(p in h for p in pats):
                return idx
        return fallback_idx

    c_tk = find_col(["티커", "종목코드", "코드"], 1)
    c_nm = find_col(["종목명", "종목"], 2)
    c_qty = find_col(["수량", "보유수량"], 3)
    c_cost = find_col(["매수금액"], 8)
    c_val = find_col(["평가금액"], 9)

    agg: dict[str, dict] = {}
    for row in rows[1:]:
        if len(row) <= max(c_tk, c_val):
            continue
        raw_tk = str(row[c_tk] or "").strip()
        if not raw_tk:
            continue
        if ":" in raw_tk:
            exch, code = raw_tk.split(":", 1)
        else:
            exch, code = "KRX", raw_tk
        exch = exch.strip().upper()
        code = code.strip()
        if exch in ("KRX", "KOSPI", "KOSDAQ") and code.isdigit():
            code = code.zfill(6)
        nm = str(row[c_nm] or "").strip() if c_nm < len(row) else code
        qty = _parse_sheet_num(row[c_qty] if c_qty < len(row) else 0)
        cost = _parse_sheet_num(row[c_cost] if c_cost < len(row) else 0)
        val = _parse_sheet_num(row[c_val] if c_val < len(row) else 0)
        if qty <= 0 and val <= 0:
            continue

        key = f"{exch}:{code}"
        if key not in agg:
            ov = overrides.get(code) or overrides.get(key) or {}
            cls = _classify_position(code, nm, exch, ov)
            agg[key] = {
                "ticker": code,
                "exchange": exch,
                "name": str(ov.get("name") or nm or code).strip(),
                "common_ticker": str(ov.get("common_ticker") or "").strip().zfill(6) if ov.get("common_ticker") else "",
                "path": str(ov.get("path") or "").strip(),
                "qty": 0.0,
                "cost_krw": 0.0,
                "value_krw": 0.0,
                "fs_basis": str(ov.get("fs_basis") or "CFS").strip().upper(),
                "share_swap_exempt": bool(ov.get("share_swap_exempt", False)),
                "note": str(ov.get("note") or "").strip(),
                **cls,
            }
        agg[key]["qty"] += qty
        agg[key]["cost_krw"] += cost
        agg[key]["value_krw"] += val

    return sorted(agg.values(), key=lambda x: -x["value_krw"])

def _parse_xlsx_tab_rows(xlsx_bytes: bytes, tab_name: str) -> list[list]:
    """순수 표준 라이브러리로 xlsx 바이트에서 지정 탭 행 추출 (gspread 폴백용)."""
    NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
          "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships"}
    with zipfile.ZipFile(io.BytesIO(xlsx_bytes)) as z:
        ss = []
        if "xl/sharedStrings.xml" in z.namelist():
            root = ET.fromstring(z.read("xl/sharedStrings.xml"))
            for si in root.findall("m:si", NS):
                ss.append("".join(t.text or "" for t in si.iter(f"{{{NS['m']}}}t")))
        wb = ET.fromstring(z.read("xl/workbook.xml"))
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        rmap = {r.attrib["Id"]: r.attrib["Target"] for r in rels}
        sheets = [(sh.attrib["name"], "xl/" + rmap[sh.attrib[f"{{{NS['r']}}}id"]].lstrip("/"))
                  for sh in wb.findall(".//m:sheet", NS)]
        target = next((tgt for nm, tgt in sheets if nm == tab_name), sheets[0][1] if sheets else None)
        if not target:
            return []
        sroot = ET.fromstring(z.read(target))
        rows = []
        for row in sroot.findall(".//m:sheetData/m:row", NS):
            cols = {}
            for c in row.findall("m:c", NS):
                col_l = "".join(ch for ch in c.attrib.get("r", "") if ch.isalpha())
                ci = sum((ord(ch) - 64) * (26 ** i) for i, ch in enumerate(reversed(col_l))) - 1
                t = c.attrib.get("t", "")
                v = c.find("m:v", NS)
                if v is not None and v.text is not None:
                    cols[ci] = ss[int(v.text)] if t == "s" else v.text
            if cols:
                mx = max(cols) + 1
                rows.append([cols.get(i, "") for i in range(mx)])
        return rows

def fetch_google_sheet_positions(sheet_id: str, tab_name: str, overrides: dict) -> tuple[list[dict], str]:
    """Colab 본인 계정 OAuth로 비공개 Google 스프레드시트 읽기 전용 조회 (gspread + Drive Export 이중화)."""
    if not sheet_id:
        return [], "PORTFOLIO_SHEET_ID 미설정"
    if not IN_COLAB:
        return [], "Colab 환경 아님"
    try:
        from google.colab import auth
        import google.auth
        auth.authenticate_user()
        scopes = [
            "https://www.googleapis.com/auth/spreadsheets.readonly",
            "https://www.googleapis.com/auth/drive.readonly",
        ]
        creds, _ = google.auth.default(scopes=scopes)
    except Exception as e:
        msg = f"Google 인증 실패({type(e).__name__}: {e})"
        print(f"[주의] {msg}")
        return [], msg

    # 1차 시도: gspread 직접 읽기
    err1 = ""
    try:
        import gspread
        gc = gspread.authorize(creds)
        sh = gc.open_by_key(sheet_id)
        try:
            ws = sh.worksheet(tab_name)
        except Exception:
            ws = sh.get_worksheet(0)
        rows = ws.get_all_values()
        parsed = parse_sheet_rows_aggregated(rows, overrides)
        if parsed:
            return parsed, f"gspread 정상 연동 ({len(parsed)}종목)"
        err1 = "시트에 유효한 보유종목 행이 없음"
    except Exception as e:
        err1 = f"gspread({type(e).__name__}: {e})"

    # 2차 폴백: Google Drive Export API로 xlsx 메모리 다운로드 후 파싱
    try:
        from google.auth.transport.requests import AuthorizedSession
        sess = AuthorizedSession(creds)
        url = f"https://www.googleapis.com/drive/v3/files/{sheet_id}/export"
        resp = sess.get(url, params={"mimeType": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"}, timeout=30)
        if resp.status_code == 200 and resp.content[:2] == b"PK":
            rows = _parse_xlsx_tab_rows(resp.content, tab_name)
            parsed = parse_sheet_rows_aggregated(rows, overrides)
            if parsed:
                return parsed, f"Drive Export 정상 연동 ({len(parsed)}종목)"
        msg = f"시트 조회 실패 ({err1} / DriveExport HTTP {resp.status_code})"
    except Exception as e2:
        msg = f"시트 조회 실패 ({err1} / Export {type(e2).__name__})"
    print(f"[주의] {msg} -> Drive YAML 모드로 대체합니다.")
    return [], msg

def load_my_portfolio() -> dict:
    """Colab Secrets / 비공개 Drive YAML / 비공개 Google 스프레드시트에서 개인 포트폴리오 로드."""
    default_pf = {
        "sheet_id": "",
        "sheet_tab": SCREENER_CONFIG["SHEET_TAB_NAME"],
        "basket_targets": {
            "A": {"name": "국내 배당성장", "target": 0.50, "low": 0.45, "high": 0.55},
            "B": {"name": "반도체·AI",     "target": 0.20, "low": 0.15, "high": 0.25},
            "C": {"name": "미국 지수",     "target": 0.30, "low": 0.25, "high": 0.35},
        },
        "kr_basket_ratio_of_total": 0.50,
        "single_stock_cap": 0.15,
        "financial_cap": 0.35,
        "ai_cap": 0.30,
        "new_cash_krw": 0,
        "include_sell_proceeds": False,
        "max_new_picks": 4,
        "prefer_liquidity_eok": SCREENER_CONFIG["PREFER_LIQUIDITY_EOK"],
        "overrides": {},
        "all_positions": [],   # 전체 계좌 종목 합산 (A+B+C, 계좌명 제거됨)
        "holdings": [],        # 한국 개별주만 추출 (정기 검진 대상)
        "watchlist": [],
        "_source": "없음 (전 시장 스크리닝 모드)",
        "_from_sheet": False,
    }

    # 1) YAML 설정 로드 (또는 템플릿 자동 생성)
    raw_pf = {}
    yaml_loaded = False
    if isinstance(INLINE_PORTFOLIO, dict):
        raw_pf = INLINE_PORTFOLIO
        yaml_loaded = True
    elif PORTFOLIO_PATH is not None:
        if not PORTFOLIO_PATH.exists():
            PORTFOLIO_PATH.write_text(PORTFOLIO_YAML_TEMPLATE, encoding="utf-8")
            print(f"[안내] 비공개 포트폴리오 설정 템플릿을 Drive에 생성했습니다: {PORTFOLIO_PATH}")
        else:
            try:
                raw_pf = yaml.safe_load(PORTFOLIO_PATH.read_text(encoding="utf-8")) or {}
                yaml_loaded = True
            except Exception as e:
                print(f"[주의] {PORTFOLIO_PATH.name} 파싱 오류({type(e).__name__}: {e})")

    pf = dict(default_pf)
    if isinstance(raw_pf, dict):
        for k in ("kr_basket_ratio_of_total", "single_stock_cap", "financial_cap", "ai_cap", "prefer_liquidity_eok"):
            if raw_pf.get(k) is not None:
                pf[k] = float(raw_pf[k])
        for k in ("new_cash_krw", "max_new_picks"):
            if raw_pf.get(k) is not None:
                pf[k] = int(float(raw_pf[k]))
        if raw_pf.get("include_sell_proceeds") is not None:
            pf["include_sell_proceeds"] = bool(raw_pf["include_sell_proceeds"])
        if raw_pf.get("sheet_tab"):
            pf["sheet_tab"] = str(raw_pf["sheet_tab"]).strip()
        if isinstance(raw_pf.get("basket_targets"), dict):
            bt = dict(default_pf["basket_targets"])
            for bk, bv in raw_pf["basket_targets"].items():
                if isinstance(bv, dict):
                    bt[str(bk).upper()] = {
                        "name": str(bv.get("name", bk)),
                        "target": float(bv.get("target", 0.33)),
                        "low": float(bv.get("low", 0.0)),
                        "high": float(bv.get("high", 1.0)),
                    }
            pf["basket_targets"] = bt
        if isinstance(raw_pf.get("overrides"), dict):
            pf["overrides"] = {str(k).strip().zfill(6) if str(k).strip().isdigit() else str(k).strip(): v for k, v in raw_pf["overrides"].items() if isinstance(v, dict)}
        pf["watchlist"] = [
            str(w["ticker"] if isinstance(w, dict) else w).strip().zfill(6)
            for w in (raw_pf.get("watchlist") or [])
            if (w.get("ticker") if isinstance(w, dict) else w)
        ]

    # 2) 스프레드시트 ID 탐색 (Colab Secrets 우선 -> 환경변수 -> Drive YAML)
    sid = ""
    sid_src = "미설정"
    if IN_COLAB and SCREENER_CONFIG["USE_GOOGLE_SHEET"]:
        try:
            from google.colab import userdata
            for secret_key in ("PORTFOLIO_SHEET_ID", "PORTFOLIO_SHEET_URL", "SHEET_ID"):
                try:
                    raw_s = userdata.get(secret_key)
                except Exception:
                    raw_s = ""
                if raw_s and str(raw_s).strip():
                    sid = _extract_sheet_id(raw_s)
                    sid_src = f"Colab Secret({secret_key})"
                    break
        except Exception:
            pass
    if not sid and os.environ.get("PORTFOLIO_SHEET_ID"):
        sid = _extract_sheet_id(os.environ.get("PORTFOLIO_SHEET_ID", ""))
        sid_src = "환경변수"
    if not sid and isinstance(raw_pf, dict) and raw_pf.get("sheet_id"):
        sid = _extract_sheet_id(raw_pf.get("sheet_id") or "")
        sid_src = "Drive YAML"
    pf["sheet_id"] = sid
    pf["_sheet_diag"] = f"ID소스={sid_src}"

    # 3) 스프레드시트 연동 시도 -> 실패/미설정 시 YAML 수동 holdings 폴백
    if sid and SCREENER_CONFIG["USE_GOOGLE_SHEET"]:
        positions, diag = fetch_google_sheet_positions(sid, pf["sheet_tab"], pf["overrides"])
        pf["_sheet_diag"] = f"{sid_src} -> {diag}"
        if positions:
            pf["all_positions"] = positions
            pf["holdings"] = [p for p in positions if (not p["is_etf"]) and (not p["is_foreign"])]
            pf["_source"] = "비공개 Google 스프레드시트 (계좌명 비노출 집계)"
            pf["_from_sheet"] = True
            return pf

    # YAML 수동 입력 폴백
    clean_h = []
    if isinstance(raw_pf, dict):
        for h in (raw_pf.get("holdings") or []):
            if isinstance(h, str):
                h = {"ticker": h}
            if not isinstance(h, dict) or not h.get("ticker"):
                continue
            tk = str(h["ticker"]).strip().zfill(6)
            ov = pf["overrides"].get(tk) or {}
            nm = str(h.get("name") or ov.get("name") or "").strip()
            cls = _classify_position(tk, nm, "KRX", {**ov, **{k: h[k] for k in ("basket", "track", "ai", "etf") if k in h}})
            ctk = str(h.get("common_ticker") or ov.get("common_ticker") or "").strip()
            ctk = ctk.zfill(6) if ctk else ""
            rec = {
                "ticker": tk, "exchange": "KRX", "name": nm, "common_ticker": ctk,
                "path": str(h.get("path") or ov.get("path") or "").strip(),
                "qty": float(h.get("qty") or 0), "cost_krw": float(h.get("cost_krw") or 0),
                "value_krw": float(h.get("value_krw") or 0),
                "fs_basis": str(h.get("fs_basis") or ov.get("fs_basis") or "CFS").strip().upper(),
                "share_swap_exempt": bool(h.get("share_swap_exempt", ov.get("share_swap_exempt", False))),
                "note": str(h.get("note") or "").strip(),
                **cls,
            }
            clean_h.append(rec)
    if clean_h:
        pf["all_positions"] = clean_h
        pf["holdings"] = [p for p in clean_h if not p["is_etf"]]
        pf["_source"] = f"비공개 Drive YAML ({PORTFOLIO_PATH.name})" if PORTFOLIO_PATH else "INLINE_PORTFOLIO"
    elif yaml_loaded:
        pf["_source"] = "Drive YAML (보유 종목 미입력 -> 전 시장 스크리닝)"
    return pf

MY_PF = load_my_portfolio()
active_guards = [k for k in ["USE_YIELD_GUARD", "USE_DPS_CAGR_FLOOR", "USE_LIQUIDITY_GUARD", "USE_CRASH_EPS_GUARD", "USE_VAL_CAP"] if SCREENER_CONFIG[k]]
print(f"[설정] FY{SCREENER_CONFIG['FISCAL_YEAR']} | 대상: {SCREENER_CONFIG['MARKETS']} 시총 {SCREENER_CONFIG['MIN_MCAP_EOK']:,}억 원 이상")
print(f"[설정] 금액 마스킹: {'켜짐(비중만 표시)' if SCREENER_CONFIG['MASK_AMOUNTS_IN_REPORT'] else '꺼짐(비공개 리포트에 금액 표시)'}")
print(f"[시트 상태] {MY_PF.get('_sheet_diag', '-')}")
print(f"[포트폴리오] 소스: {MY_PF['_source']} | 전체 {len(MY_PF['all_positions'])}종목 (한국 개별주 {len(MY_PF['holdings'])} · 관심 {len(MY_PF['watchlist'])}) | 신규투자금 {MY_PF['new_cash_krw']:,}원")


In [ ]:
# ==============================================================================
# 3. 1단계: 코스피·코스닥 전 종목 시총 목록 수집 (무인증) + DART 고유번호 매핑
# ==============================================================================
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-screener-colab)"}
_EXCLUDE_RE = re.compile(r"(ETF|ETN|리츠|스팩|SPAC|인프라|선박투자|펀드|KODEX|TIGER|ACE|KBSTAR|SOL|ARIRANG|HANARO|PLUS|RISE)")
_PATH_HINT = [
    ("securities", r"증권"),
    ("financial_holding", r"캐피탈|카드|투자|한국금융지주|메리츠금융"),
    ("bank", r"은행|금융지주|신한지주|우리금융|하나금융|BNK|DGB|iM금융|JB금융|KB금융|금융$"),
    ("insurance", r"생명(?!과학|공학|자원)|화재|보험|손보|코리안리"),
]

def guess_path(name: str) -> str:
    for p, pat in _PATH_HINT:
        if re.search(pat, str(name)):
            return p
    return "general"

def _num(v):
    if v in (None, "", "-", "－", "N/A"):
        return None
    s = str(v).replace(",", "").replace("배", "").replace("원", "").replace("%", "").strip()
    try:
        x = float(s)
        return None if (math.isnan(x) or math.isinf(x)) else x
    except ValueError:
        return None

def _num_dash0(v):
    return 0.0 if str(v).strip() in ("-", "－") else _num(v)

def fetch_naver_market_list(market: str) -> pd.DataFrame:
    rows, page = [], 1
    while True:
        r = requests.get(
            f"https://m.stock.naver.com/api/stocks/marketValue/{market}",
            params={"page": page, "pageSize": 100}, headers=_HEADERS, timeout=20
        )
        r.raise_for_status()
        stocks = r.json().get("stocks") or []
        if not stocks:
            break
        for s in stocks:
            rows.append({
                "ticker": str(s["itemCode"]).zfill(6),
                "name": s["stockName"],
                "close": _num(s.get("closePrice")),
                "mcap_eok": _num(s.get("marketValue")),
                "end_type": s.get("stockEndType"),
                "market": market,
            })
        page += 1
        time.sleep(0.1)
    df = pd.DataFrame(rows)
    df["is_pref"] = df["ticker"].str[-1] != "0"
    df["excluded"] = (df["end_type"] != "stock") | df["name"].str.contains(_EXCLUDE_RE)
    return df

def fetch_corp_codes() -> dict[str, str]:
    cache_p = CACHE_DIR / "corp_codes.json"
    if cache_p.exists():
        return json.loads(cache_p.read_text(encoding="utf-8"))
    r = requests.get("https://opendart.fss.or.kr/api/corpCode.xml", params={"crtfc_key": DART_API_KEY}, timeout=90)
    if r.status_code != 200:
        raise RuntimeError(f"OpenDART corpCode.xml HTTP {r.status_code}")
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        xml_bytes = z.read(z.namelist()[0])
    root = ET.fromstring(xml_bytes)
    mapping = {}
    for el in root.iter("list"):
        sc = (el.findtext("stock_code") or "").strip()
        cc = (el.findtext("corp_code") or "").strip()
        if sc and cc:
            mapping[sc.zfill(6)] = cc
    cache_p.write_text(json.dumps(mapping), encoding="utf-8")
    return mapping

def load_rf_rates() -> tuple[dict[int, float], float]:
    # 연도별 국고채 10년물 연평균 금리 (Common_Everything Public CSV)
    rf_map = {2015:0.0230, 2016:0.0175, 2017:0.0228, 2018:0.0250, 2019:0.0170,
              2020:0.0150, 2021:0.0207, 2022:0.0337, 2023:0.0364, 2024:0.0322, 2025:0.0289}
    for br in SCREENER_CONFIG["GITHUB_BRANCHES"]:
        url = f"https://raw.githubusercontent.com/{SCREENER_CONFIG['GITHUB_REPO']}/{br}/{SCREENER_CONFIG['DATA_SUBPATH']}/kr_treasury_10y_annual.csv"
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200:
                df = pd.read_csv(io.StringIO(r.text))
                rf_map = {int(row["연도"]): float(row["국고채10년_연평균_퍼센트"]) / 100.0 for _, row in df.iterrows()}
                break
        except Exception:
            pass
    # 현재 국고채 10년물 금리 (네이버)
    cur_rf = rf_map[max(rf_map)]
    try:
        rj = requests.get("https://api.stock.naver.com/marketindex/bond/KR10YT=RR", headers=_HEADERS, timeout=10).json()
        cur_rf = float(rj["closePrice"]) / 100.0
    except Exception:
        pass
    return rf_map, cur_rf

print("[1단계] 코스피·코스닥 전 종목 목록 및 DART 고유번호 수집 중...")
all_mkt = pd.concat([fetch_naver_market_list(m) for m in SCREENER_CONFIG["MARKETS"]], ignore_index=True).drop_duplicates("ticker")
CORP_MAP = fetch_corp_codes()
RF_MAP, CUR_RF = load_rf_rates()

uni = all_mkt[(~all_mkt["excluded"]) & (~all_mkt["is_pref"]) & (all_mkt["mcap_eok"].fillna(0) >= SCREENER_CONFIG["MIN_MCAP_EOK"])].copy()
uni["corp_code"] = uni["ticker"].map(CORP_MAP)
uni = uni[uni["corp_code"].notna()].reset_index(drop=True)
uni["path"] = uni["name"].map(guess_path)

print(f"[완료] 전체 상장 {len(all_mkt):,}종목 -> 보통주·시총 {SCREENER_CONFIG['MIN_MCAP_EOK']:,}억↑ & DART 매핑 완료: {len(uni):,}종목")
print(f"       시장별: {uni['market'].value_counts().to_dict()} | 국고채 10년물 현재 금리: {CUR_RF:.2%}")

# 보유 종목 및 관심 종목의 보통주 티커·DART 고유번호·현재가 자동 매핑 (우선주·소형주 모두 지원)
ALL_MKT_MAP = {row["ticker"]: row.to_dict() for _, row in all_mkt.iterrows()}

def resolve_extra_targets(pf: dict) -> tuple[list[dict], list[dict]]:
    resolved_h, extra_scan = [], []
    seen_common = set(uni["ticker"].tolist())

    for h in pf["holdings"]:
        tk = h["ticker"]
        m_info = ALL_MKT_MAP.get(tk) or {}
        is_pref = tk[-1] != "0"
        ctk = h["common_ticker"] or (tk[:5] + "0" if is_pref else tk)
        if ctk not in CORP_MAP and tk in CORP_MAP:
            ctk = tk
        c_info = ALL_MKT_MAP.get(ctk) or m_info
        nm = h["name"] or m_info.get("name") or c_info.get("name") or tk
        path_val = h["path"] or guess_path(nm)
        rec = {
            **h,
            "name": nm,
            "is_pref": is_pref,
            "common_ticker": ctk,
            "corp_code": CORP_MAP.get(ctk),
            "path": path_val,
            "market": m_info.get("market") or c_info.get("market") or "KOSPI",
            "close_hold": m_info.get("close") or c_info.get("close"),
            "mcap_eok": c_info.get("mcap_eok") or m_info.get("mcap_eok") or 0.0,
        }
        resolved_h.append(rec)
        if ctk and CORP_MAP.get(ctk) and ctk not in seen_common:
            seen_common.add(ctk)
            extra_scan.append({
                "ticker": ctk,
                "name": c_info.get("name") or nm,
                "close": c_info.get("close") or m_info.get("close"),
                "mcap_eok": c_info.get("mcap_eok") or 0.0,
                "end_type": "stock",
                "market": rec["market"],
                "is_pref": False,
                "excluded": False,
                "corp_code": CORP_MAP[ctk],
                "path": path_val,
            })

    for wtk in pf["watchlist"]:
        ctk = wtk[:5] + "0" if (wtk[-1] != "0" and (wtk[:5] + "0") in CORP_MAP) else wtk
        if ctk in CORP_MAP and ctk not in seen_common:
            seen_common.add(ctk)
            c_info = ALL_MKT_MAP.get(ctk) or ALL_MKT_MAP.get(wtk) or {}
            nm = c_info.get("name") or ctk
            extra_scan.append({
                "ticker": ctk,
                "name": nm,
                "close": c_info.get("close"),
                "mcap_eok": c_info.get("mcap_eok") or 0.0,
                "end_type": "stock",
                "market": c_info.get("market") or "KOSPI",
                "is_pref": False,
                "excluded": False,
                "corp_code": CORP_MAP[ctk],
                "path": guess_path(nm),
            })
    return resolved_h, extra_scan

MY_HOLDINGS, EXTRA_SCAN_ROWS = resolve_extra_targets(MY_PF)
EXTRA_COMMON_SET = {h["common_ticker"] for h in MY_HOLDINGS if h.get("corp_code")} | {
    (wtk[:5] + "0" if (wtk[-1] != "0" and (wtk[:5] + "0") in CORP_MAP) else wtk) for wtk in MY_PF["watchlist"]
}
if EXTRA_SCAN_ROWS:
    print(f"[보유·관심 추가] 시총 기준 외 보유·관심 보통주 {len(EXTRA_SCAN_ROWS)}종목을 강제 점검 대상에 포함했습니다: {[r['name'] for r in EXTRA_SCAN_ROWS]}")


# 보유 종목별 특수 옵션(별도 OFS 재무제표, 총환원형 트랙, 주식교환 희석 예외)을 보통주 티커 기준으로 매핑
HOLD_OPT_BY_COMMON = {}
for h in MY_HOLDINGS:
    ctk = h.get("common_ticker")
    if ctk:
        HOLD_OPT_BY_COMMON[ctk] = {
            "fs_basis": str(h.get("fs_basis") or "CFS").upper(),
            "track": str(h.get("track") or "dividend").lower(),
            "share_swap_exempt": bool(h.get("share_swap_exempt", False)),
        }


In [ ]:
# ==============================================================================
# 4. 2단계: OpenDART 배당 이력 고속 1차 선별 (3년 단위 4회 호출 -> 12년치 확보)
# ==============================================================================
DART_BASE = "https://opendart.fss.or.kr/api"
DART_CALLS = {"count": 0, "limit_hit": False}

def dart_get(endpoint: str, corp_code: str, bsns_year: int, extra: dict | None = None) -> list[dict]:
    if DART_CALLS["limit_hit"]:
        return []
    key_tag = f"{endpoint}_{corp_code}_{bsns_year}" + (f"_{extra.get('fs_div','')}" if extra else "")
    cache_p = CACHE_DIR / f"{key_tag}.json"
    if cache_p.exists():
        return json.loads(cache_p.read_text(encoding="utf-8"))

    params = {"crtfc_key": DART_API_KEY, "corp_code": corp_code, "bsns_year": str(bsns_year), "reprt_code": "11011"}
    if extra:
        params.update(extra)

    for attempt in range(4):
        try:
            DART_CALLS["count"] += 1
            r = requests.get(f"{DART_BASE}/{endpoint}.json", params=params, timeout=30)
            if r.status_code != 200:
                time.sleep(1.5 * (attempt + 1))
                continue
            j = r.json()
            st = j.get("status")
            if st == "020":   # 일일 요청 제한 초과
                DART_CALLS["limit_hit"] = True
                print("[경고] OpenDART 일일 요청 한도(020)에 도달했습니다. 지금까지 수집된 캐시로 진행합니다.")
                return []
            if st == "013":   # 데이터 없음
                cache_p.write_text("[]", encoding="utf-8")
                return []
            if st == "000":
                rows = j.get("list") or []
                cache_p.write_text(json.dumps(rows, ensure_ascii=False), encoding="utf-8")
                return rows
            return []
        except Exception:
            time.sleep(1.0 * (attempt + 1))
    return []

def _stock_class(s: str) -> str | None:
    s = str(s or "")
    if "보통" in s or "의결권 있는" in s or "의결권있는" in s:
        return "common"
    if "우선" in s or "종류주" in s or "의결권 없는" in s or "의결권없는" in s:
        return "pref"
    return None

def parse_alot_12y(corp_code: str, last_year: int) -> dict[int, dict]:
    '''last_year, -3, -6, -9 총 4회 호출로 최근 12개 사업연도 배당·EPS·순이익·배당성향 추출.'''
    by_year: dict[int, dict] = {}
    call_years = [y for y in (last_year, last_year - 3, last_year - 6, last_year - 9) if y >= 2015]
    for cy in call_years:
        rows = dart_get("alotMatter", corp_code, cy)
        if not rows:
            continue
        # 결산일이 여러 개면 가장 늦은 결산일만 사용
        st_dates = sorted({str(r.get("stlm_dt") or "").strip() for r in rows if str(r.get("stlm_dt") or "").strip() not in ("", "-")})
        if len(st_dates) > 1:
            rows = [r for r in rows if str(r.get("stlm_dt") or "").strip() == st_dates[-1]]
        cfs_ni = any("당기순이익" in str(r.get("se", "")) and "연결" in str(r.get("se", "")) for r in rows)

        for col, lag in (("thstrm", 0), ("frmtrm", 1), ("lwfr", 2)):
            y = cy - lag
            if y in by_year and by_year[y].get("dps_common") is not None:
                continue
            out = by_year.setdefault(y, {})
            period_sum, total_dash0 = 0.0, False
            for r in rows:
                se = str(r.get("se", ""))
                knd_raw = str(r.get("stock_knd") or "").strip()
                knd = "common" if knd_raw in ("", "-", "nan", "None") else _stock_class(knd_raw)
                v = _num(r.get(col))
                if "주당 현금배당금" in se and re.search(r"결산|중간|분기", knd_raw):
                    if v is not None:
                        period_sum += v
                    continue
                if "액면" in se and v is not None:
                    out["face_value"] = v
                elif "당기순이익" in se and "별도" not in se and not ("개별" in se and cfs_ni):
                    if v is not None: out["ni_reported"] = v * 1e6
                elif "주당순이익" in se and v is not None:
                    out["eps_reported"] = v
                elif "현금배당금총액" in se:
                    if v is not None: out["div_total_reported"] = v * 1e6
                    total_dash0 = _num_dash0(r.get(col)) == 0.0
                elif "현금배당성향" in se and v is not None:
                    out["payout_reported"] = v
                elif "주당 현금배당금" in se and knd == "common" and out.get("dps_common") is None:
                    out["dps_common"] = v
            if out.get("dps_common") is None and period_sum > 0:
                out["dps_common"] = period_sum
            if out.get("dps_common") is None and out.get("eps_reported") is not None and total_dash0:
                out["dps_common"] = 0.0
                out["payout_reported"] = 0.0

    # 액면분할 보정 (최신 액면가 기준)
    latest_fv = next((by_year[y]["face_value"] for y in sorted(by_year, reverse=True) if by_year[y].get("face_value")), None)
    if latest_fv and latest_fv > 0:
        for y, d in by_year.items():
            fv = d.get("face_value")
            if fv and fv > 0 and abs(fv - latest_fv) > 0.5:
                ratio = latest_fv / fv
                for k in ("dps_common", "eps_reported"):
                    if d.get(k) is not None:
                        d[k] = d[k] * ratio
    return by_year

def avg3_dict(by_year: dict, col: str, end: int):
    vals = [by_year[y][col] for y in (end - 2, end - 1, end) if y in by_year and by_year[y].get(col) is not None]
    return sum(vals) / len(vals) if len(vals) >= 2 else None

def cagr3_dict(by_year: dict, col: str, end: int, span: int):
    a, b = avg3_dict(by_year, col, end - span), avg3_dict(by_year, col, end)
    if a is None or a <= 0 or b is None:
        return None
    return -1.0 if b <= 0 else (b / a) ** (1.0 / span) - 1.0

def stage1_worker(row_dict: dict, last_year: int) -> dict:
    corp = row_dict["corp_code"]
    by_year = parse_alot_12y(corp, last_year)
    min_yrs = SCREENER_CONFIG["MIN_CONSECUTIVE_DIV_YEARS"]
    paid_run = 0
    for y in range(last_year, last_year - 13, -1):
        if (by_year.get(y) or {}).get("dps_common") and by_year[y]["dps_common"] > 0:
            paid_run += 1
        else:
            break
    g10 = cagr3_dict(by_year, "dps_common", last_year, 10)
    g5 = cagr3_dict(by_year, "dps_common", last_year, 5)
    e5 = cagr3_dict(by_year, "eps_reported", last_year, 5)
    cuts5 = 0
    for y in range(last_year - 4, last_year + 1):
        a = (by_year.get(y - 1) or {}).get("dps_common")
        b = (by_year.get(y) or {}).get("dps_common")
        if a is not None and b is not None and b < a - min(0.5, abs(a) * 1e-6):
            cuts5 += 1
    # 3년 합 배당성향
    d3 = [by_year.get(y, {}).get("div_total_reported") for y in (last_year - 2, last_year - 1, last_year)]
    n3 = [by_year.get(y, {}).get("ni_reported") for y in (last_year - 2, last_year - 1, last_year)]
    p3 = (sum(d3) / sum(n3) * 100.0) if (all(v is not None for v in d3 + n3) and sum(n3) > 0) else None

    pass1 = (
        paid_run >= min_yrs
        and g5 is not None and g5 > 0
        and (g10 is None or g10 > 0)
        and (e5 is None or e5 >= 0)
        and cuts5 <= SCREENER_CONFIG["MAX_CUTS_5Y"]
        and p3 is not None and 0 < p3 <= SCREENER_CONFIG["MAX_PAYOUT_3Y"]
    )
    return {
        **row_dict, "pass1": pass1, "paid_run": paid_run,
        "dps_g10": g10, "dps_g5": g5, "eps_g5": e5, "cuts_5y": cuts5, "payout3_div": p3,
        "dps_last": (by_year.get(last_year) or {}).get("dps_common"),
        "by_year_div": by_year,
    }

FY = SCREENER_CONFIG["FISCAL_YEAR"]
CKPT1_NAME = f"ckpt_stage1_FY{FY}.json.gz"
ckpt1 = load_checkpoint(CKPT1_NAME)
# JSON 직렬화 시 dict key가 문자열이 되므로 by_year_div 연도 키를 int로 복원
def _restore_int_keys(d: dict) -> dict:
    return {int(k): v for k, v in (d or {}).items()}

stage1_map = {}
for t, rec in ckpt1.items():
    r_copy = dict(rec)
    r_copy["by_year_div"] = _restore_int_keys(r_copy.get("by_year_div"))
    stage1_map[t] = r_copy

scan_df = pd.concat([uni, pd.DataFrame(EXTRA_SCAN_ROWS)], ignore_index=True).drop_duplicates("ticker") if EXTRA_SCAN_ROWS else uni
todo_rows = [row.to_dict() for _, row in scan_df.iterrows() if row["ticker"] not in stage1_map]
if stage1_map:
    print(f"[이어받기] 기존 Drive/캐시 완료분 {len(stage1_map):,}종목 재사용 -> 남은 {len(todo_rows):,}종목만 스캔합니다!")
else:
    print(f"[2단계] {len(uni):,}개 종목 OpenDART 1차 배당 이력 스캔 시작 (FY{FY})...")

t0 = time.time()
try:
    with ThreadPoolExecutor(max_workers=SCREENER_CONFIG["WORKERS"]) as ex:
        futs = [ex.submit(stage1_worker, r, FY) for r in todo_rows]
        for idx, fut in enumerate(as_completed(futs), 1):
            res = fut.result()
            stage1_map[res["ticker"]] = res
            if idx % 50 == 0 or idx == len(futs):
                save_checkpoint(CKPT1_NAME, stage1_map)
                n_ok = sum(1 for x in stage1_map.values() if x["ticker"] in set(uni["ticker"]) and x["pass1"])
                print(f"  진행 {len(stage1_map):,}/{len(uni):,} (신규 {idx:,}/{len(futs):,}) | 1차 통과 {n_ok}종목 | Drive 저장됨 | DART {DART_CALLS['count']:,}회 ({time.time()-t0:.0f}초)")
finally:
    if todo_rows:
        save_checkpoint(CKPT1_NAME, stage1_map)

# 최신 시총·종가 정보는 scan_df 기준으로 동기화 (보유·관심 종목은 1차 통과 여부와 무관하게 2차 강제 수집!)
scan_lookup = {row["ticker"]: row.to_dict() for _, row in scan_df.iterrows()}
uni_tickers = set(uni["ticker"].tolist())
SURVIVORS = []
FORCED_HOLD_ITEMS = []
for t, row_now in scan_lookup.items():
    rec = stage1_map.get(t)
    if not rec:
        continue
    merged = {**rec, "close": row_now["close"], "mcap_eok": row_now["mcap_eok"], "path": row_now["path"], "_in_uni": t in uni_tickers}
    if rec.get("pass1") and t in uni_tickers:
        SURVIVORS.append(merged)
    elif t in EXTRA_COMMON_SET:
        merged["_forced_extra"] = True
        FORCED_HOLD_ITEMS.append(merged)

STAGE2_TARGETS = SURVIVORS + [x for x in FORCED_HOLD_ITEMS if x["ticker"] not in {s["ticker"] for s in SURVIVORS}]
print(f"[완료] 1차 배당 이력 통과: {len(SURVIVORS)}종목 / {len(uni):,}종목 (+ 보유·관심 강제점검 {len(STAGE2_TARGETS)-len(SURVIVORS)}종목) ({time.time()-t0:.1f}초)")


In [ ]:
# ==============================================================================
# 5. 3단계: 1차 통과 종목 정밀 수집 (DART 재무·주식·자사주 + 네이버 시세·컨센서스)
# ==============================================================================
_FCHART_URL = "https://fchart.stock.naver.com/sise.nhn"
_ITEM_RE = re.compile(r'<item data="([^"]+)"')

def fetch_naver_daily(ticker: str, count: int = 3200) -> pd.DataFrame:
    for attempt in range(3):
        try:
            r = requests.get(_FCHART_URL, params={"symbol": ticker, "timeframe": "day", "count": count, "requestType": 0}, headers=_HEADERS, timeout=20)
            r.raise_for_status()
            rows = [(pd.Timestamp(d), float(o), float(h), float(l), float(c), float(v))
                    for d, o, h, l, c, v in (m.group(1).split("|") for m in _ITEM_RE.finditer(r.content.decode("euc-kr", errors="ignore")))]
            df = pd.DataFrame(rows, columns=["date", "open", "high", "low", "close", "volume"]).set_index("date")
            df = df[df["close"] > 0].sort_index()
            if not df.empty:
                return df
        except Exception:
            time.sleep(0.4 * (attempt + 1))
    return pd.DataFrame()

def fetch_naver_consensus(ticker: str, target_fy: int) -> dict | None:
    try:
        r = requests.get(f"https://m.stock.naver.com/api/stock/{ticker}/finance/annual", headers=_HEADERS, timeout=15)
        info = r.json().get("financeInfo") or {}
        cons_cols = [t for t in info.get("trTitleList", []) if t.get("isConsensus") == "Y"]
        if not cons_cols:
            return None
        col = cons_cols[0]
        m = re.match(r"(\d{4})", str(col.get("title") or ""))
        if m and int(m.group(1)) != target_fy:
            return None
        key = col["key"]
        out = {"year": col["title"]}
        name_map = {"EPS": "eps", "BPS": "bps", "주당배당금": "dps"}
        for row in info.get("rowList", []):
            k = name_map.get(row.get("title"))
            if k:
                out[k] = _num(((row.get("columns") or {}).get(key) or {}).get("value"))
        return out if (out.get("eps") and out["eps"] > 0) else None
    except Exception:
        return None

def extract_fs_year(rows: list[dict], col: str) -> dict:
    '''OpenDART fnlttSinglAcntAll 응답에서 핵심 재무 계정 추출.'''
    def find(ids, nm_pat, sj_pref=None):
        cand = rows
        if sj_pref:
            sub = [r for r in rows if r.get("sj_div") in sj_pref]
            if sub:
                cand = sub
        for aid in ids:
            for r in cand:
                rid = re.sub(r"^(ifrs-full|ifrs|dart)_", "", str(r.get("account_id") or ""))
                if rid == aid:
                    v = _num(r.get(col))
                    if v is not None:
                        return v
        if nm_pat:
            rx = re.compile(nm_pat)
            for r in cand:
                if rx.match(str(r.get("account_nm") or "").strip()):
                    v = _num(r.get(col))
                    if v is not None:
                        return v
        return None

    op = find(["OperatingIncomeLoss"], r"^영업(이익|손실|순손익|이익\(손실\))$", ("IS", "CIS"))
    ni_p = find(["ProfitLossAttributableToOwnersOfParent"], r"^지배(기업|주주)(의)?\s*(소유주|지분)?(?!.*(포괄|주당|중단|계속)).*(당기)?순(이익|손익|손실)", ("IS", "CIS"))
    ni = find(["ProfitLoss"], r"^당기순이익(\(손실\))?$", ("IS", "CIS"))
    cfo = find(["CashFlowsFromUsedInOperatingActivities"], r"^영업활동.*현금흐름", ("CF",))
    capex_ppe = find(["PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities", "PurchaseOfPropertyPlantAndEquipment"], r".*유형자산의?\s*(취득|증가)$", ("CF",))
    capex_int = find(["PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities"], r".*무형자산의?\s*취득$", ("CF",))
    tp = find(["PaymentsForPurchaseOfTreasuryShares", "AcquisitionOfTreasuryShares", "PurchaseOfTreasuryShares"], r".*(자기주식|자사주)(의)?\s*(취득|매입)", ("CF",))
    eq_p = find(["EquityAttributableToOwnersOfParent"], r"지배(기업|주주).*지분$", ("BS",))
    eq = find(["Equity"], r"^자\s*본\s*총\s*계$", ("BS",))
    cash = find(["CashAndCashEquivalents", "CashEquivalents"], r"^현금\s*및\s*현금성\s*자산$|^현금성자산$", ("BS",))
    st_fin = find(["ShorttermDepositsNotClassifiedAsCashEquivalents"], r"^단기금융상품$", ("BS",))
    inv_aff = find(["InvestmentsInSubsidiariesJointVenturesAndAssociates", "InvestmentAccountedForUsingEquityMethod"], r"^(종속기업|관계기업|공동기업).*투자", ("BS",))
    int_exp = find(["InterestExpense"], r"^이자비용$", ("IS", "CIS"))
    int_paid = find(["InterestPaidClassifiedAsOperatingActivities"], r"^이자의?\s*지급", ("CF",))
    dep = find(["DepreciationAndAmortisationExpense", "AdjustmentsForDepreciationAndAmortisationExpense"], r"^감가상각비", ("CF", "IS", "CIS"))

    # 차입금·사채 합산 (재무상태표)
    debt_exc = re.compile(r"채권|자산|투자|대여|할인|할증|발행비|이자$|미지급|조정|충당|보증|리스|파생")
    debt_seen, debt_sum, debt_found = set(), 0.0, False
    for r in rows:
        if r.get("sj_div") != "BS":
            continue
        nm = str(r.get("account_nm") or "").replace(" ", "")
        if ("차입" in nm or "사채" in nm or nm.startswith("유동성장기부채")) and not debt_exc.search(nm):
            v = _num(r.get(col))
            if v is not None and (nm, v) not in debt_seen:
                debt_seen.add((nm, v))
                debt_sum += abs(v)
                debt_found = True
    debt = debt_sum if debt_found else (0.0 if (cash is not None and op and abs(int_exp or int_paid or 0) < abs(op) * 0.05) else None)

    capex = (abs(capex_ppe) + abs(capex_int or 0.0)) if capex_ppe is not None else None
    fcf = (cfo - capex) if (cfo is not None and capex is not None) else None
    net_debt = (debt - cash - (st_fin or 0.0)) if (debt is not None and cash is not None) else None
    ebitda = (op + abs(dep)) if (op is not None and dep is not None) else None
    interest = int_exp if int_exp else int_paid
    icr = (op / abs(interest)) if (op is not None and interest and abs(interest) > 0) else None
    nde = (net_debt / ebitda) if (net_debt is not None and ebitda is not None and ebitda > 0) else None
    invested = (eq + net_debt - (inv_aff or 0.0)) if (eq is not None and net_debt is not None) else None
    roic = (op * (1 - 0.24) / invested) if (op is not None and invested is not None and invested > 0) else None
    roe_ni = ni_p if ni_p is not None else ni
    roe_eq = eq_p if (ni_p is not None and eq_p) else eq
    roe = (roe_ni / roe_eq) if (roe_ni is not None and roe_eq and roe_eq > 0) else None

    return {
        "op": op, "ni_p": ni_p, "ni": ni, "fcf": fcf, "tp": abs(tp) if tp is not None else 0.0,
        "eq_p": eq_p, "eq": eq, "debt": debt, "net_debt": net_debt, "ebitda": ebitda,
        "icr": icr, "nde": nde, "roic": roic, "roe": roe, "int": abs(interest) if interest else None,
        "has_fs": (ni is not None or eq is not None),
    }

def collect_stage2_one(item: dict, fy: int) -> dict:
    corp, ticker = item["corp_code"], item["ticker"]
    # 1) 재무제표 수집 (기본 CFS 우선, 보유 옵션 fs_basis=='OFS' 지정 시 별도 OFS 강제 수집)
    opt_h = HOLD_OPT_BY_COMMON.get(ticker, {})
    req_fs_basis = opt_h.get("fs_basis", "CFS")
    fs_by_year = {}
    for cy in [y for y in (fy, fy - 3, fy - 6, fy - 9) if y >= 2015]:
        if req_fs_basis == "OFS":
            rows = dart_get("fnlttSinglAcntAll", corp, cy, {"fs_div": "OFS"})
            if not rows:
                rows = dart_get("fnlttSinglAcntAll", corp, cy, {"fs_div": "CFS"})
        else:
            rows = dart_get("fnlttSinglAcntAll", corp, cy, {"fs_div": "CFS"})
            if not rows:
                rows = dart_get("fnlttSinglAcntAll", corp, cy, {"fs_div": "OFS"})
        if not rows:
            continue
        for col, lag in (("thstrm_amount", 0), ("frmtrm_amount", 1), ("bfefrmtrm_amount", 2)):
            y = cy - lag
            if y not in fs_by_year or not fs_by_year[y].get("has_fs"):
                ext = extract_fs_year(rows, col)
                if ext["has_fs"]:
                    fs_by_year[y] = ext

    # 2) 주식 총수 (FY 및 FY-5)
    sh_by_year = {}
    for y in (fy, fy - 5):
        rows = dart_get("stockTotqySttus", corp, y)
        if not rows:
            continue
        rcp = max(str(r.get("rcept_no") or "") for r in rows)
        rows = [r for r in rows if str(r.get("rcept_no") or "") == rcp]
        oc, op_sh = 0.0, 0.0
        for r in rows:
            k = _stock_class(r.get("se", ""))
            v = _num(r.get("distb_stock_co"))
            if k == "common" and v: oc += v
            elif k == "pref" and v: op_sh += v
        if oc > 0:
            sh_by_year[y] = {"common": oc, "pref": op_sh}

    # 3) 자기주식 취득·소각 (최근 5년)
    ts_by_year = {}
    for y in range(fy - 4, fy + 1):
        rows = dart_get("tesstkAcqsDspsSttus", corp, y)
        if not rows:
            continue
        acq, canc = 0.0, 0.0
        for r in rows:
            if _stock_class(r.get("stock_knd", "")) == "common" and str(r.get("acqs_mth1")) == "총계":
                acq += _num_dash0(r.get("change_qy_acqs")) or 0.0
                canc += _num_dash0(r.get("change_qy_incnr")) or 0.0
        ts_by_year[y] = {"acq": acq, "canc": canc}

    # 4) 네이버 일별 주가 + 컨센서스
    px_df = fetch_naver_daily(ticker)
    cons = fetch_naver_consensus(ticker, fy + 1)

    return {**item, "fs": fs_by_year, "shares": sh_by_year, "ts": ts_by_year, "px_df": px_df, "cons": cons, "fs_basis": req_fs_basis}

CKPT2_NAME = f"ckpt_stage2_dart_FY{FY}.json.gz"
ckpt2 = load_checkpoint(CKPT2_NAME)
dart2_map = {}
for t, rec in ckpt2.items():
    dart2_map[t] = {
        "fs": _restore_int_keys(rec.get("fs")),
        "shares": _restore_int_keys(rec.get("shares")),
        "ts": _restore_int_keys(rec.get("ts")),
    }

def _ckpt2_key(ticker: str) -> str:
    return f"{ticker}_OFS" if HOLD_OPT_BY_COMMON.get(ticker, {}).get("fs_basis") == "OFS" else ticker

def collect_stage2_resumable(item: dict, fy: int) -> dict:
    t = item["ticker"]
    ck_key = _ckpt2_key(t)
    req_b = HOLD_OPT_BY_COMMON.get(t, {}).get("fs_basis", "CFS")
    cached_d = dart2_map.get(ck_key)
    if cached_d is not None:
        px_df = fetch_naver_daily(t)
        cons = fetch_naver_consensus(t, fy + 1)
        return {**item, **cached_d, "px_df": px_df, "cons": cons, "fs_basis": req_b, "_from_ckpt": True, "_ckpt_key": ck_key}
    res = collect_stage2_one(item, fy)
    res["_from_ckpt"] = False
    res["_ckpt_key"] = ck_key
    return res

reused2 = sum(1 for it in STAGE2_TARGETS if _ckpt2_key(it["ticker"]) in dart2_map)
if reused2:
    print(f"[3단계 이어받기] 2차 DART 재무 {reused2}/{len(STAGE2_TARGETS)}종목 재사용 -> 남은 {len(STAGE2_TARGETS)-reused2}종목 DART 수집 + 최신 시세/컨센서스 갱신 중...")
else:
    print(f"[3단계] 1차 통과 {len(STAGE2_TARGETS)}종목 정밀 수집 중...")

t0 = time.time()
STAGE2_DATA = []
new_cnt2 = 0
try:
    with ThreadPoolExecutor(max_workers=SCREENER_CONFIG["WORKERS"]) as ex:
        futs = [ex.submit(collect_stage2_resumable, it, FY) for it in STAGE2_TARGETS]
        for idx, fut in enumerate(as_completed(futs), 1):
            r = fut.result()
            STAGE2_DATA.append(r)
            if not r.get("_from_ckpt"):
                dart2_map[r.get("_ckpt_key", r["ticker"])] = {"fs": r["fs"], "shares": r["shares"], "ts": r["ts"]}
                new_cnt2 += 1
            if (new_cnt2 > 0 and new_cnt2 % 15 == 0) or idx == len(futs):
                save_checkpoint(CKPT2_NAME, dart2_map)
                print(f"  진행 {idx}/{len(futs)} | Drive 저장됨 | 누적 DART 호출 {DART_CALLS['count']:,}회 ({time.time()-t0:.0f}초)")
finally:
    if new_cnt2 > 0:
        save_checkpoint(CKPT2_NAME, dart2_map)

print(f"[완료] 2차 정밀 데이터 수집 완료 ({time.time()-t0:.1f}초, 총 DART 호출 {DART_CALLS['count']:,}회)")

# 보유 종목 중 우선주(보통주와 티커가 다른 경우)는 우선주 실제 시세도 별도 수집
PREF_PX_MAP = {}
for h in MY_HOLDINGS:
    if h["is_pref"] and h["ticker"] not in PREF_PX_MAP:
        PREF_PX_MAP[h["ticker"]] = fetch_naver_daily(h["ticker"], count=400)
if PREF_PX_MAP:
    print(f"[우선주 시세] 보유 우선주 {len(PREF_PX_MAP)}종목 실제 일별 주가 수집 완료")


In [ ]:
# ==============================================================================
# 6. 4단계: 1~9번 정량 판정 + 재투자 우선순위 + 선택적 보완 필터 적용
# ==============================================================================
ERP, TAX = 0.05, 0.24

def judge_screener_one(d: dict, fy: int) -> dict | None:
    px_df = d["px_df"]
    if px_df is None or px_df.empty:
        return None
    cur_close = float(px_df["close"].iloc[-1])
    turnover_60d = float((px_df["close"] * px_df["volume"]).tail(60).mean())
    w52 = px_df.loc[px_df.index >= px_df.index[-1] - pd.Timedelta(days=365)]
    dd_52w = (cur_close / float(w52["high"].max()) - 1.0) if not w52.empty else 0.0
    ye_px = px_df["close"].groupby(px_df.index.year).last().to_dict()

    path = d["path"]
    div_y, fs_y, sh_y, ts_y = d["by_year_div"], d["fs"], d["shares"], d["ts"]

    # 1번 배당 이력
    v1 = "FAIL" if (d["paid_run"] < 7 or d["cuts_5y"] >= 2) else (
        "PASS" if (d["paid_run"] >= 10 and d["cuts_5y"] == 0 and (d["dps_g5"] or 0) > 0 and (d["dps_g10"] is None or d["dps_g10"] > 0)) else "WARN"
    )

    # 3년 합 총환원율 & 배당성향
    W3 = (fy - 2, fy - 1, fy)
    d3 = [div_y.get(y, {}).get("div_total_reported") for y in W3]
    e3 = [(fs_y.get(y, {}).get("ni_p") if fs_y.get(y, {}).get("ni_p") is not None else div_y.get(y, {}).get("ni_reported")) for y in W3]
    if all(v is not None for v in d3 + e3) and sum(e3) > 0:
        b3 = sum(fs_y.get(y, {}).get("tp", 0.0) for y in W3)
        c3 = sum(ts_y.get(y, {}).get("canc", 0.0) for y in W3)
        a3 = sum(ts_y.get(y, {}).get("acq", 0.0) for y in W3)
        s_rat = min(1.0, c3 / a3) if a3 > 0 else (1.0 if c3 > 0 else 0.0)
        ret3_num = sum(d3) + b3 * s_rat
        tr_3y = ret3_num / sum(e3) * 100.0
        p3 = sum(d3) / sum(e3) * 100.0
    else:
        ret3_num, tr_3y, p3 = None, None, d["payout3_div"]

    # 2번 EPS 성장
    g_eps, g_dps = d["eps_g5"], d["dps_g5"]
    if g_eps is None or g_dps is None: v2 = "NA"
    elif g_eps < 0 and g_dps > 0: v2 = "FAIL"
    elif g_eps > 0 and (g_dps <= g_eps + 0.03 or (p3 is not None and p3 < 30.0)): v2 = "PASS"
    else: v2 = "WARN"

    # 3번 FCF / 순이익 꾸준함
    vals3 = [fs_y.get(y, {}).get("fcf") if path == "general" else (fs_y.get(y, {}).get("ni_p") or div_y.get(y, {}).get("ni_reported"))
             for y in range(fy - 9, fy + 1)]
    k3 = [v for v in vals3 if v is not None]
    if sum(1 for v in k3 if v < 0) >= 3: v3 = "FAIL"
    elif len(k3) >= 5 and sum(1 for v in k3 if v > 0) / len(k3) >= 0.8: v3 = "PASS"
    elif len(k3) >= 5: v3 = "WARN"
    else: v3 = "NA"

    # 4번 ROIC-WACC / ROE-COE
    spreads = []
    for y in range(fy - 4, fy + 1):
        coe = RF_MAP.get(y, CUR_RF) + ERP
        f = fs_y.get(y) or {}
        if path == "general":
            if f.get("roic") is None: continue
            wacc = coe
            sh_tot = (sh_y.get(fy, {}).get("common", 0) + sh_y.get(fy, {}).get("pref", 0))
            mcap_y = ye_px.get(y, 0) * sh_tot
            debt_y = f.get("debt") or 0.0
            if mcap_y > 0:
                kd = (min(0.15, f["int"] / debt_y) if (debt_y > 0 and f.get("int")) else 0.04) * (1 - TAX)
                wacc = coe * (mcap_y / (mcap_y + debt_y)) + kd * (debt_y / (mcap_y + debt_y))
            spreads.append((y, f["roic"] - wacc))
        else:
            if f.get("roe") is not None: spreads.append((y, f["roe"] - coe))
    p_thr, f_thr = (0.03, 0.0) if path == "general" else (0.01, -0.01)
    avg5_sp = sum(s for _, s in spreads) / len(spreads) if len(spreads) >= 3 else None
    l3_sp = [s for y, s in spreads if y >= fy - 2]
    if len(l3_sp) == 3 and all(s < f_thr for s in l3_sp): v4 = "FAIL"
    elif avg5_sp is not None:
        r2 = [s for y, s in spreads if y >= fy - 1]
        v4 = "PASS" if (avg5_sp >= p_thr and (not r2 or sum(r2)/len(r2) >= avg5_sp - 0.03)) else "WARN"
    else: v4 = "NA"

    # 5번 주주환원 수준
    if tr_3y is None or p3 is None: v5 = "NA"
    elif tr_3y < 20.0 or p3 > 75.0 or (path != "general" and tr_3y > 100.0): v5 = "FAIL"
    elif tr_3y >= 30.0 and p3 <= 60.0 and (path == "general" or tr_3y <= 66.7): v5 = "PASS"
    else: v5 = "WARN"

    # 6번 자사주·희석
    s_now = (sh_y.get(fy) or {}).get("common")
    s_old = (sh_y.get(fy - 5) or {}).get("common")
    if not s_now or not s_old: v6 = "NA"
    else:
        chg = s_now / s_old - 1.0
        canc5 = sum(ts_y.get(y, {}).get("canc", 0.0) for y in range(fy - 4, fy + 1))
        acq5 = sum(ts_y.get(y, {}).get("acq", 0.0) for y in range(fy - 4, fy + 1))
        _hopt = HOLD_OPT_BY_COMMON.get(d["ticker"], {})
        if _hopt.get("share_swap_exempt") and canc5 > 0:
            v6 = "PASS"
        elif chg >= 0.03:
            v6 = "FAIL"
        elif (chg < -0.01 and canc5 > 0) or (-0.01 <= chg <= 0.01 and (acq5 == 0 or canc5 >= acq5)):
            v6 = "PASS"
        else:
            v6 = "WARN"

    # 7번 환원 감당 (일반만 판정)
    if path != "general": v7 = "참고"
    else:
        fcfs = [fs_y.get(y, {}).get("fcf") for y in W3]
        if ret3_num is None or any(v is None for v in fcfs): v7 = "NA"
        elif sum(fcfs) <= 0: v7 = "FAIL" if ret3_num > 0 else "PASS"
        else:
            cov = ret3_num / sum(fcfs)
            v7 = "FAIL" if cov > 1.0 else ("WARN" if cov > 2/3 else "PASS")

    # 8번 재무건전성 (금융은 사업보고서 본문 규제자본비율 별도 확인 필요)
    if path != "general":
        v8 = "NA(금융본문)"
    else:
        f_end = fs_y.get(fy) or {}
        nd, nde, icr, eb = f_end.get("net_debt"), f_end.get("nde"), f_end.get("icr"), f_end.get("ebitda")
        if nd is not None and nd <= 0: v8 = "WARN" if (eb is not None and eb <= 0) else "PASS"
        elif (icr is not None and icr < 3.0) or (nde is not None and nde > 3.0): v8 = "FAIL"
        elif nde is not None and icr is not None: v8 = "PASS" if (nde < 2.0 and icr >= 5.0) else "WARN"
        else: v8 = "NA"

    _hopt = HOLD_OPT_BY_COMMON.get(d["ticker"], {})
    is_tr_track = (_hopt.get("track") == "total_return")
    if is_tr_track:
        # 총환원형 트랙(rules.md): 1·2번 현금배당 기준은 참고용, 3~6번(+일반 7~8번)으로 품질 종합 판정
        counted = [v3, v4, v5, v6] + ([v7, v8] if path == "general" else [])
    else:
        counted = [v1, v2, v3, v4, v5, v6] + ([v7, v8] if path == "general" else [])
    n_fail = counted.count("FAIL")
    n_warn = counted.count("WARN")
    n_na = sum(1 for v in counted if str(v).startswith("NA"))
    if n_fail >= 1 or n_warn >= 5:
        overall = "미부합"
    elif n_warn <= 2 and n_na == 0:
        overall = "원칙 부합" if path == "general" else "원칙 부합(금융자본확인)"
    elif n_na >= 3:
        overall = "판정 보류"
    else:
        overall = "부분 부합"

    # 9번 밸류에이션 (선행 컨센서스 우선 -> FY 기준)
    cons = d["cons"]
    eps_fy = (div_y.get(fy) or {}).get("eps_reported")
    dps_fy = d["dps_last"]
    eq_fy = (fs_y.get(fy) or {}).get("eq_p") or (fs_y.get(fy) or {}).get("eq")
    sh_tot_fy = (sh_y.get(fy, {}).get("common", 0) + sh_y.get(fy, {}).get("pref", 0))
    bps_fy = (eq_fy / sh_tot_fy) if (eq_fy and sh_tot_fy > 0) else None

    if cons:
        val_basis = f"선행({cons['year']})"
        c_per = cur_close / cons["eps"] if cons.get("eps") and cons["eps"] > 0 else None
        c_pbr = cur_close / cons["bps"] if cons.get("bps") and cons["bps"] > 0 else None
        c_yld = cons["dps"] / cur_close if cons.get("dps") and cons["dps"] > 0 else None
    else:
        val_basis = f"FY{fy}"
        c_per = cur_close / eps_fy if (eps_fy and eps_fy > 0) else None
        c_pbr = cur_close / bps_fy if (bps_fy and bps_fy > 0) else None
        c_yld = dps_fy / cur_close if (dps_fy and dps_fy > 0) else None

    h_per, h_pbr, h_yld = [], [], []
    for y in range(fy - 9, fy + 1):
        px = ye_px.get(y)
        if not px or px <= 0: continue
        e = (div_y.get(y) or {}).get("eps_reported")
        dp = (div_y.get(y) or {}).get("dps_common")
        eq = (fs_y.get(y) or {}).get("eq_p") or (fs_y.get(y) or {}).get("eq")
        b = (eq / sh_tot_fy) if (eq and sh_tot_fy > 0) else None
        if e and e > 0: h_per.append(px / e)
        if b and b > 0: h_pbr.append(px / b)
        if dp and dp > 0: h_yld.append(dp / px)

    devs_raw, devs_cap, cheap, dear = [], [], 0, 0
    cap_lim = SCREENER_CONFIG["VAL_CAP"]
    for cv, hv, hc in ((c_per, h_per, False), (c_pbr, h_pbr, False), (c_yld, h_yld, True)):
        if cv is not None and len(hv) >= 5:
            med = statistics.median(hv)
            if med > 0:
                dev = (cv / med - 1.0) if hc else -(cv / med - 1.0)
                devs_raw.append(dev)
                devs_cap.append(max(-cap_lim, min(cap_lim, dev)))
                if dev >= 0: cheap += 1
                if dev < -0.20: dear += 1

    v9 = "NA" if len(devs_raw) < 2 else ("PASS" if cheap >= 2 else ("WARN" if dear >= 2 else "중간"))
    val_raw = sum(devs_raw) / len(devs_raw) if len(devs_raw) >= 2 else None
    val_cap = sum(devs_cap) / len(devs_cap) if len(devs_cap) >= 2 else None
    cur_yld_fy = dps_fy / cur_close if (dps_fy and cur_close > 0) else None

    # 선택적 보완 필터 판정
    eps_prev = (div_y.get(fy - 1) or {}).get("eps_reported")
    eps_down = bool(eps_fy is not None and eps_prev is not None and eps_fy < eps_prev)
    g_notes = []
    if SCREENER_CONFIG["USE_YIELD_GUARD"] and (cur_yld_fy is None or not (SCREENER_CONFIG["MIN_YIELD"] <= cur_yld_fy <= SCREENER_CONFIG["MAX_YIELD"])):
        g_notes.append(f"배당률범위외({(cur_yld_fy or 0):.1%})")
    if SCREENER_CONFIG["USE_DPS_CAGR_FLOOR"] and (g_dps is None or g_dps < SCREENER_CONFIG["MIN_DPS_CAGR_5Y"]):
        g_notes.append(f"DPS성장<{SCREENER_CONFIG['MIN_DPS_CAGR_5Y']:.0%}")
    if SCREENER_CONFIG["USE_LIQUIDITY_GUARD"] and turnover_60d < SCREENER_CONFIG["MIN_TURNOVER_60D"]:
        g_notes.append(f"저유동성({turnover_60d/1e8:.1f}억)")
    if SCREENER_CONFIG["USE_CRASH_EPS_GUARD"] and dd_52w <= SCREENER_CONFIG["MAX_DD_52W_WITH_EPS_DROP"] and eps_down:
        g_notes.append(f"급락+EPS감소({dd_52w:.1%})")

    return {
        "ticker": d["ticker"], "name": d["name"], "market": d["market"], "path": path, "mcap_eok": d["mcap_eok"],
        "overall": overall, "n_fail": n_fail, "n_warn": n_warn, "n_na": n_na,
        "v1": v1, "v2": v2, "v3": v3, "v4": v4, "v5": v5, "v6": v6, "v7": v7, "v8": v8, "v9": v9,
        "val_basis": val_basis, "val_raw": val_raw, "val_cap": val_cap,
        "dps_g5": g_dps, "eps_g5": g_eps, "tr_3y": tr_3y, "payout_3y": p3,
        "spread_5y": avg5_sp, "close": cur_close, "yield_fy": cur_yld_fy, "yield_val": c_yld,
        "per": c_per, "pbr": c_pbr, "dd_52w": dd_52w, "turnover_60d_eok": turnover_60d / 1e8,
        "guard_pass": len(g_notes) == 0, "guard_note": ", ".join(g_notes) if g_notes else "통과",
        # 매도규칙 4절 A 진단 필드 + 1~9번 항목별 실제 수치 요약
        "fs_basis": d.get("fs_basis", "CFS"),
        "fail_items": [nm for nm, vv in zip(
            ["1번 배당이력", "2번 EPS성장", "3번 FCF/자본", "4번 스프레드", "5번 환원수준", "6번 자사주", "7번 환원감당", "8번 건전성"],
            [v1, v2, v3, v4, v5, v6, v7, v8],
        ) if vv == "FAIL" and (path == "general" or not nm.startswith(("7번", "8번"))) and (not is_tr_track or not nm.startswith(("1번", "2번")))],
        "item_details": {
            "1": f"연속배당 {d['paid_run']}년 · 5년삭감 {d['cuts_5y']}회",
            "2": f"EPS {g_eps:+.1%} vs DPS {g_dps:+.1%}" if (g_eps is not None and g_dps is not None) else "자료 부족",
            "3": (f"10년 FCF>0 {sum(1 for v in k3 if v > 0)}/{len(k3)}년" if path == "general" else f"자본증가 {sum(1 for v in k3 if v > 0)}/{len(k3)}년"),
            "4": f"5년 평균 {avg5_sp:+.1%} (통과 {p_thr:+.0%})" if avg5_sp is not None else "자료 부족",
            "5": f"총환원 {tr_3y:.1f}% · 성향 {p3:.1f}%" if (tr_3y is not None and p3 is not None) else "자료 부족",
            "6": ("주식교환 제외 · 소각 지속" if _hopt.get("share_swap_exempt") else (f"주식수 {(s_now/s_old-1):+.1%}" if (s_now and s_old) else "자료 부족")),
            "7": ("금융 참고" if path != "general" else (f"환원/FCF {ret3_num/sum(fcfs):.0%}" if (ret3_num is not None and all(v is not None for v in fcfs) and sum(fcfs) > 0) else ("3년 FCF 음수" if all(v is not None for v in fcfs) else "자료 부족"))),
            "8": ("금융본문 확인" if path != "general" else ("순현금" if ((fs_y.get(fy) or {}).get("net_debt") is not None and (fs_y.get(fy) or {}).get("net_debt") <= 0) else (f"순부채/EBITDA {((fs_y.get(fy) or {}).get('nde') or 0):.2f}배 · ICR {((fs_y.get(fy) or {}).get('icr') or 0):.1f}배"))),
            "9": f"할인 {val_raw:+.0%} ({val_basis})" if val_raw is not None else "자료 부족",
        },
        "dps_y_m2": (div_y.get(fy - 2) or {}).get("dps_common"),
        "dps_y_m1": (div_y.get(fy - 1) or {}).get("dps_common"),
        "dps_y_0": (div_y.get(fy) or {}).get("dps_common"),
        "liq_ok": (turnover_60d / 1e8) >= float(MY_PF.get("prefer_liquidity_eok", SCREENER_CONFIG["PREFER_LIQUIDITY_EOK"])),
        "liq_badge": "양호" if (turnover_60d / 1e8) >= float(MY_PF.get("prefer_liquidity_eok", SCREENER_CONFIG["PREFER_LIQUIDITY_EOK"])) else f"주의(<{float(MY_PF.get('prefer_liquidity_eok', SCREENER_CONFIG['PREFER_LIQUIDITY_EOK'])):.0f}억)",
        "is_fin": path != "general",
        "_in_uni": bool(d.get("_in_uni", True)),
        "_pass1": bool(d.get("pass1", False)),
    }

JUDGED_ALL = [x for d in STAGE2_DATA if (x := judge_screener_one(d, FY)) is not None]
JUDGED_MAP = {x["ticker"]: x for x in JUDGED_ALL}
# 시장 스크리닝 표본은 기존 1차 통과 모집단(SURVIVORS) 기준으로 유지 (보유 강제점검 종목과 섞이지 않게)
JUDGED = [x for x in JUDGED_ALL if x["_in_uni"] and x["_pass1"]]
val_col = "val_cap" if SCREENER_CONFIG["USE_VAL_CAP"] else "val_raw"

HELD_COMMON_SET = {h["common_ticker"] for h in MY_HOLDINGS}
WATCH_COMMON_SET = {
    (wtk[:5] + "0" if (wtk[-1] != "0" and (wtk[:5] + "0") in CORP_MAP) else wtk)
    for wtk in MY_PF["watchlist"]
}

# 원칙 부합(일반 + 금융자본확인) 우선순위 정렬: 밸류 순위 + DPS 5년 성장 순위 합
TOP_POOL = [x for x in JUDGED if x["overall"].startswith("원칙 부합") and x["guard_pass"] and x[val_col] is not None]
if TOP_POOL:
    vals = [x[val_col] for x in TOP_POOL]
    grs = [x["dps_g5"] if x["dps_g5"] is not None else -99 for x in TOP_POOL]
    for x in TOP_POOL:
        vr = 1 + sum(1 for v in vals if v > x[val_col])
        gr = 1 + sum(1 for g in grs if g > (x["dps_g5"] if x["dps_g5"] is not None else -99))
        x["rank_sum"] = vr + gr
        x["hold_tag"] = "보유중" if x["ticker"] in HELD_COMMON_SET else ("관심" if x["ticker"] in WATCH_COMMON_SET else "신규")
    TOP_POOL.sort(key=lambda x: (x["rank_sum"], -(x[val_col] or -9), -(x["spread_5y"] or -9)))
    def _build_strength_caution(x: dict) -> tuple[str, str]:
        st, ca = [], []
        vv = x.get(val_col)
        gg = x.get("dps_g5")
        yy = x.get("yield_fy")
        tr = x.get("tr_3y")
        sp = x.get("spread_5y")
        dd = x.get("dd_52w")
        if vv is not None and vv >= 0.40: st.append(f"초저평가({vv:+.0%})")
        elif vv is not None and vv >= 0.15: st.append(f"저평가({vv:+.0%})")
        if gg is not None and gg >= 0.30: st.append(f"배당고성장({gg:+.0%})")
        elif gg is not None and gg >= 0.12: st.append(f"배당성장({gg:+.0%})")
        if yy is not None and yy >= 0.055: st.append(f"고배당({yy:.1%})")
        if tr is not None and tr >= 45.0: st.append(f"최상위환원({tr:.0f}%)")
        elif tr is not None and tr >= 35.0: st.append(f"우수환원({tr:.0f}%)")
        if sp is not None and sp >= 0.20: st.append(f"압도적수익성(+{sp:.0%})")
        elif sp is not None and sp >= 0.08: st.append(f"고수익성(+{sp:.0%})")
        if dd is not None and dd >= -0.15: st.append("주가안정")
        if not st: st.append("균형잡힌 원칙부합")

        if dd is not None and dd <= -0.45: ca.append(f"52주 급락({dd:.0%}) 사유 점검")
        if not x.get("liq_ok", True): ca.append(f"거래대금 얇음({x['turnover_60d_eok']:.1f}억)")
        if x.get("v9") == "WARN": ca.append("현재 주가 부담(9번 WARN)")
        if x.get("is_fin"): ca.append("사업보고서 규제자본비율 확인")
        if not ca: ca.append("특이 리스크 없음")
        return " · ".join(st[:3]), " · ".join(ca[:2])

    for r_i, x in enumerate(TOP_POOL, 1):
        x["overall_rank"] = r_i
        x["strength"], x["caution"] = _build_strength_caution(x)

ALL_DF = pd.DataFrame(JUDGED)
TOP_DF = pd.DataFrame(TOP_POOL)
TOP_NONFIN_DF = TOP_DF[~TOP_DF["is_fin"]].reset_index(drop=True) if not TOP_DF.empty else pd.DataFrame()
TOP_FIN_DF = TOP_DF[TOP_DF["is_fin"]].reset_index(drop=True) if not TOP_DF.empty else pd.DataFrame()

print(f"■ 스크리닝 완료 요약 (FY{FY}, 기준일 {date.today().isoformat()})")
if not ALL_DF.empty:
    print(ALL_DF["overall"].value_counts().to_string())
print(f"\n■ 최종 원칙 부합 후보: {len(TOP_DF)}종목 (비금융 {len(TOP_NONFIN_DF)} · 금융 {len(TOP_FIN_DF)})")

PATH_KO = {
    "general": "일반",
    "bank": "은행",
    "insurance": "보험",
    "securities": "증권",
    "financial_holding": "금융지주",
}

def _overall_short(v: str) -> str:
    return "원칙 부합*" if str(v) == "원칙 부합(금융자본확인)" else str(v)

def fmt_table(df: pd.DataFrame, include_rank: bool = False) -> pd.DataFrame:
    if df.empty:
        return df
    cols = {}
    if include_rank and "overall_rank" in df.columns:
        cols["순위"] = df["overall_rank"].astype(int)
    if "hold_tag" in df.columns:
        cols["구분"] = df["hold_tag"]
    cols.update({
        "종목명": df["name"], "티커": df["ticker"], "시장": df["market"],
        "업종": df["path"].map(lambda p: PATH_KO.get(str(p), str(p))),
        "종합": df["overall"].map(_overall_short),
        "가격(9번)": df["v9"],
        "밸류할인": df[val_col].map(lambda v: "-" if pd.isna(v) else f"{v:+.0%}"),
        "DPS 5년": df["dps_g5"].map(lambda v: "-" if pd.isna(v) else f"{v:+.1%}"),
        "EPS 5년": df["eps_g5"].map(lambda v: "-" if pd.isna(v) else f"{v:+.1%}"),
        "배당률(FY)": df["yield_fy"].map(lambda v: "-" if pd.isna(v) else f"{v:.2%}"),
        "총환원율(3년)": df["tr_3y"].map(lambda v: "-" if pd.isna(v) else f"{v:.1f}%"),
        "스프레드": df["spread_5y"].map(lambda v: "-" if pd.isna(v) else f"{v:+.1%}"),
        "52주낙폭": df["dd_52w"].map(lambda v: f"{v:.1%}"),
        "거래대금(억)": df["turnover_60d_eok"].map(lambda v: f"{v:,.1f}"),
        "유동성": df["liq_badge"],
        "시총(억)": df["mcap_eok"].map(lambda v: f"{v:,.0f}"),
    })
    if "strength" in df.columns:
        cols["핵심 강점"] = df["strength"]
    if "caution" in df.columns:
        cols["체크포인트"] = df["caution"]
    return pd.DataFrame(cols)

TOP_VIEW = fmt_table(TOP_DF, include_rank=True)
TOP_NONFIN_VIEW = fmt_table(TOP_NONFIN_DF, include_rank=True)
TOP_FIN_VIEW = fmt_table(TOP_FIN_DF, include_rank=True)
# 예비 후보: FAIL 0이지만 WARN 3~4로 '부분 부합' 또는 '판정 보류'인 종목 (내년 승격 후보)
RESERVE_DF = ALL_DF[ALL_DF["overall"].isin(["부분 부합", "판정 보류"])].sort_values(
    ["n_warn", val_col], ascending=[True, False]
).reset_index(drop=True) if not ALL_DF.empty else pd.DataFrame()
RESERVE_VIEW = fmt_table(RESERVE_DF) if not RESERVE_DF.empty else pd.DataFrame()
ALL_VIEW = fmt_table(ALL_DF.sort_values(["n_fail", "n_warn", "n_na", val_col], ascending=[True, True, True, False])) if not ALL_DF.empty else pd.DataFrame()
display(TOP_VIEW.head(25))


In [ ]:
# ==============================================================================
# 7. 전체 바구니(A/B/C)·3대 상한선 진단 · 내 보유 한국주 정밀 검진 · 4~5월 매수 배분표
# ==============================================================================
def _fmt_krw_man(val: float, mask: bool) -> str:
    return "비공개" if mask else f"{val/1e4:,.0f}만 원"

def _fmt_krw_eok(val: float, mask: bool) -> str:
    if mask:
        return "비공개"
    return f"{val/1e8:.2f}억 원" if abs(val) >= 1e8 else f"{val/1e4:,.0f}만 원"

def analyze_whole_portfolio(pf: dict, judged_map: dict) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """전체 포지션(A+B+C, 계좌명 미포함) 기준 바구니 비중 및 3대 상한선(단일15%·금융35%·AI30%) 계산."""
    positions = pf.get("all_positions") or []
    tot_val = sum(float(p.get("value_krw") or 0) for p in positions)
    tot_cost = sum(float(p.get("cost_krw") or 0) for p in positions)
    if not positions or tot_val <= 0:
        return pd.DataFrame(), pd.DataFrame(), {"has_whole": False, "tot_val": 0.0, "tot_cost": 0.0}

    mask = bool(SCREENER_CONFIG["MASK_AMOUNTS_IN_REPORT"])
    b_targets = pf["basket_targets"]
    b_vals = {k: 0.0 for k in b_targets}
    fin_val = 0.0
    ai_val = 0.0
    single_cap = float(pf.get("single_stock_cap", 0.15))
    fin_cap = float(pf.get("financial_cap", 0.35))
    ai_cap = float(pf.get("ai_cap", 0.30))
    breached_single = []
    breached_tickers = set()

    for p in positions:
        v = float(p.get("value_krw") or 0)
        bk = str(p.get("basket") or "A").upper()
        b_vals[bk] = b_vals.get(bk, 0.0) + v
        ai_val += v * float(p.get("ai") or 0.0)

        if not p.get("is_etf") and not p.get("is_foreign"):
            ctk = p.get("common_ticker") or (p["ticker"][:5] + "0" if p["ticker"][-1] != "0" else p["ticker"])
            j = judged_map.get(ctk)
            path_eff = (j["path"] if j else None) or p.get("path") or guess_path(p.get("name", ""))
            if path_eff != "general":
                fin_val += v
            w = v / tot_val
            if w > single_cap + 1e-6:
                breached_single.append(f"{p['name']}({w:.1%})")
                breached_tickers.add(p["ticker"])

    basket_rows = []
    for bk, cfg in b_targets.items():
        v = b_vals.get(bk, 0.0)
        w = v / tot_val
        lo, hi, tgt = cfg["low"], cfg["high"], cfg["target"]
        st = "허용폭 초과" if w > hi + 1e-6 else ("허용폭 미달" if w < lo - 1e-6 else "정상 범위")
        basket_rows.append({
            "바구니": f"{bk}. {cfg['name']}",
            "현재 평가액": _fmt_krw_eok(v, mask),
            "현재 비중": f"{w:.1%}",
            "목표 (허용폭)": f"{tgt:.0%} ({lo:.0%}~{hi:.0%})",
            "목표 대비": f"{(w - tgt)*100:+.1f}%p",
            "판정": st,
        })

    fin_w = fin_val / tot_val
    ai_w = ai_val / tot_val
    cap_rows = [
        {
            "상한 항목": "국내 금융 업종 합계",
            "현재 비중": f"{fin_w:.1%}",
            "상한 기준": f"{fin_cap:.0%}",
            "여유 / 초과": f"{(fin_w - fin_cap)*100:+.1f}%p",
            "상태": "상한 초과 (추매 중단)" if fin_w > fin_cap + 1e-6 else "정상",
        },
        {
            "상한 항목": "AI·반도체 실제 노출",
            "현재 비중": f"{ai_w:.1%}",
            "상한 기준": f"{ai_cap:.0%}",
            "여유 / 초과": f"{(ai_w - ai_cap)*100:+.1f}%p",
            "상태": "상한 초과 (추매 중단)" if ai_w > ai_cap + 1e-6 else ("상한 임박" if ai_w >= ai_cap - 0.015 else "정상"),
        },
        {
            "상한 항목": "개별 주식 단일 상한",
            "현재 비중": f"초과 {len(breached_single)}종목" if breached_single else "모두 이내",
            "상한 기준": f"{single_cap:.0%}",
            "여유 / 초과": ", ".join(breached_single) if breached_single else "-",
            "상태": "상한 초과 (추매 중단)" if breached_single else "정상",
        },
    ]

    meta = {
        "has_whole": True,
        "tot_val": tot_val,
        "tot_cost": tot_cost,
        "tot_ret": (tot_val / tot_cost - 1.0) if tot_cost > 0 else None,
        "fin_val": fin_val,
        "fin_w": fin_w,
        "fin_breach": fin_w >= fin_cap - 1e-6,
        "ai_val": ai_val,
        "ai_w": ai_w,
        "ai_breach": ai_w >= ai_cap - 1e-6,
        "breached_single": breached_single,
        "breached_tickers": breached_tickers,
        "b_weights": {bk: b_vals.get(bk, 0.0) / tot_val for bk in b_targets},
    }
    return pd.DataFrame(basket_rows), pd.DataFrame(cap_rows), meta

def evaluate_my_holdings(holdings: list[dict], judged_map: dict, pref_px_map: dict, pf: dict, whole_meta: dict) -> tuple[pd.DataFrame, list[dict], dict]:
    if not holdings:
        return pd.DataFrame(), [], {"has_holdings": False, "has_amounts": False}

    mask = bool(SCREENER_CONFIG["MASK_AMOUNTS_IN_REPORT"])
    rows, cards = [], []
    tot_val_kr = sum(float(h.get("value_krw") or 0) for h in holdings)
    has_amounts = tot_val_kr > 0
    kr_ratio = max(0.05, min(1.0, float(pf.get("kr_basket_ratio_of_total", 0.50))))
    if whole_meta.get("has_whole") and whole_meta["tot_val"] > 0:
        est_total_asset = whole_meta["tot_val"]
    else:
        est_total_asset = (tot_val_kr / kr_ratio) if has_amounts else 0.0

    single_cap = float(pf.get("single_stock_cap", 0.15))
    fin_cap = float(pf.get("financial_cap", 0.35))
    fin_breach_global = bool(whole_meta.get("fin_breach", False))
    breached_tks = set(whole_meta.get("breached_tickers") or set())

    for h in holdings:
        tk = h["ticker"]
        ctk = h["common_ticker"]
        track = h["track"]
        j = judged_map.get(ctk)
        disp_name = h["name"] + ("(우)" if h["is_pref"] and not h["name"].endswith(("우", "우B")) else "")

        live_close = h.get("close_hold")
        dd_show = j["dd_52w"] if j else None
        if h["is_pref"] and tk in pref_px_map and not pref_px_map[tk].empty:
            pdf = pref_px_map[tk]
            live_close = float(pdf["close"].iloc[-1])
            hi52 = float(pdf["close"].tail(252).max())
            dd_show = (live_close / hi52 - 1.0) if hi52 > 0 else dd_show
        elif j and (not live_close or live_close <= 0):
            live_close = j["close"]

        w_tot = (h["value_krw"] / est_total_asset) if est_total_asset > 0 else 0.0
        is_single_breach = (tk in breached_tks) or (est_total_asset > 0 and w_tot > single_cap + 1e-6)

        if j is None:
            rows.append({
                "종목명": disp_name, "티커": tk, "트랙": track, "업종": PATH_KO.get(str(h["path"]), str(h["path"])),
                "품질종합": "수집불가", "가격(9번)": "-", "밸류할인": "-", "배당률(FY)": "-",
                "52주낙폭": "-", "매도·상태 요약": "DART 조회 실패", "액션 판정": "수동 확인",
                "평가금액": _fmt_krw_man(h["value_krw"], mask) if has_amounts else "-",
                "전체비중": f"{w_tot:.1%}" if est_total_asset > 0 else "-",
                "_is_fin": h["path"] != "general", "_value_krw": h["value_krw"], "_sell_flag": False,
                "_common": ctk, "_close": live_close or 0.0,
            })
            continue

        d_m2, d_m1, d_0 = j.get("dps_y_m2"), j.get("dps_y_m1"), j.get("dps_y_0")
        div_stopped = (d_0 is not None and d_0 <= 0)
        consec_cut_2y = (
            d_m2 is not None and d_m1 is not None and d_0 is not None
            and d_m2 > 0 and d_m1 < d_m2 * 0.99 and d_0 < d_m1 * 0.99
        )
        recent_cut_1y = (
            not consec_cut_2y and d_m1 is not None and d_0 is not None
            and d_m1 > 0 and d_0 < d_m1 * 0.99
        )
        fails = j.get("fail_items") or []
        idet = j.get("item_details") or {}
        fs_tag = "별도(OFS)" if j.get("fs_basis") == "OFS" else "연결(CFS)"
        dps_trend_str = " → ".join(f"{int(round(x)):,}원" if x is not None else "-" for x in (d_m2, d_m1, d_0))

        sell_reasons, detail_lines = [], []
        if track == "dividend":
            if div_stopped:
                sell_reasons.append("배당 중단")
                detail_lines.append(f"정기배당 중단 (최근 3년 DPS: {dps_trend_str})")
            elif consec_cut_2y:
                sell_reasons.append("정기배당 2년 연속 삭감")
                detail_lines.append(f"정기배당 2년 연속 삭감 ({dps_trend_str}) — 운용규칙 4절 A 매도 요건 충족")
            if fails:
                sell_reasons.append(f"FAIL({', '.join(fails)})")
                for fnm in fails:
                    fnum = fnm[0]
                    detail_lines.append(f"{fnm} FAIL: {idet.get(fnum, '기준 미달')} ({fs_tag} 재무 기준)")

        # 상한선까지 반영한 정확한 액션 판정!
        if track == "total_return":
            action = "별도 트랙 (총환원·추매중단)" if (is_single_breach or fin_breach_global) else "별도 트랙 (총환원)"
            sig_txt = "자사주 소각 중심 총환원형" + (" · 단일상한 초과" if is_single_breach else "") + (f" · 가격 {j['v9']}" if j["v9"] == "WARN" else "")
            sell_flag = False
            tr_txt = f"{j['tr_3y']:.1f}%" if j.get("tr_3y") is not None else "-"
            detail_lines.append(f"총환원형 트랙: 현금배당 대신 자사주 소각 정책(3년 총환원율 {tr_txt}, 4번 스프레드 {idet.get('4','-')}). 1·2번 현금배당 기준은 참고용.")
            if is_single_breach:
                detail_lines.append(f"현재 비중 {w_tot:.1%}로 단일 상한({single_cap:.0%}) 초과 — 신규 추매 중단, 내년 4월 비중 축소 검토.")
        elif track == "cycle":
            action = "별도 트랙 (사이클·추매중단)" if is_single_breach else "별도 트랙 (사이클)"
            sig_txt = "사이클 트랙 별도 관리" + (" · 단일상한 초과" if is_single_breach else "")
            sell_flag = False
            detail_lines.append(f"경기 사이클 트랙: 배당 매도규칙 대신 분기 영업이익률·재고·PBR 사이클 지표로 관리 (현재 1~8번 품질 '{_overall_short(j['overall'])}').")
            if is_single_breach:
                detail_lines.append(f"현재 비중 {w_tot:.1%}로 단일 상한({single_cap:.0%}) 초과 — 신규 추매 중단.")
        elif sell_reasons:
            action = "매도 검토"
            sig_txt = ", ".join(sell_reasons)
            sell_flag = True
        elif recent_cut_1y or not j["overall"].startswith("원칙 부합"):
            action = "관찰 (보유 유지)"
            sig_txt = f"최근 1회 배당 삭감({dps_trend_str})" if recent_cut_1y else f"품질 {j['overall']} (WARN {j['n_warn']})"
            sell_flag = False
            if recent_cut_1y:
                detail_lines.append(f"관찰 사유: 최근 1회 배당 삭감({dps_trend_str})이지만 2년 연속 삭감이 아니므로 매도 사유 아님. {fs_tag} 재무 기준 FAIL 0건 — 다음 사업연도 배당 회복 여부 확인.")
            else:
                detail_lines.append(f"관찰 사유: FAIL은 없으나 WARN {j['n_warn']}건({fs_tag} 기준)으로 보유 유지하며 실적·배당 추이 점검.")
        elif is_single_breach:
            action = "보유 유지 (단일상한·추매중단)"
            sig_txt = f"품질 양호 · 단일 상한({single_cap:.0%}) 초과"
            sell_flag = False
            detail_lines.append(f"매도 사유 없음(원칙 부합). 단 현재 비중 {w_tot:.1%}로 단일 상한 초과 상태라 신규 매수 중단.")
        elif j["is_fin"] and fin_breach_global:
            action = "보유 유지 (금융상한·추매보류)"
            sig_txt = f"품질 양호 · 국내 금융 상한({fin_cap:.0%}) 초과로 추매 보류"
            sell_flag = False
            detail_lines.append(f"매도 사유 없음(원칙 부합, 3년 총환원율 {j['tr_3y']:.1f}%). 단 전체 포트폴리오 국내 금융 비중이 {whole_meta['fin_w']:.1%}(상한 {fin_cap:.0%})로 초과 상태라 추가 매수 보류." if j.get("tr_3y") is not None else "매도 사유 없음. 국내 금융 상한 초과로 추가 매수 보류.")
        elif j["v9"] == "WARN":
            action = "보유 유지 (가격부담·추매주의)"
            sig_txt = "품질 양호 · 현재 가격 다소 비쌈"
            sell_flag = False
            detail_lines.append(f"매도 사유 없음(원칙 부합). 다만 9번 밸류에이션이 WARN({idet.get('9','-')})이라 신규 추매는 보수적 접근.")
        else:
            action = "보유 유지 (추매 가능)"
            sig_txt = "정상 (매도 사유 없음 · 상한 이내)"
            sell_flag = False
            detail_lines.append(f"매도 사유 없음(원칙 부합, {fs_tag} 기준). 9번 가격 {j['v9']}({idet.get('9','-')})로 비중 여유 시 추가 매수 가능.")

        yld_disp = (d_0 / live_close) if (d_0 and live_close and live_close > 0) else j.get("yield_fy")
        val_v = j.get(val_col)
        rows.append({
            "종목명": disp_name,
            "티커": tk,
            "트랙": {"dividend": "배당성장", "total_return": "총환원형", "cycle": "사이클"}.get(track, track),
            "업종": PATH_KO.get(str(j["path"]), str(j["path"])),
            "품질종합": _overall_short(j["overall"]),
            "가격(9번)": j["v9"],
            "밸류할인": "-" if val_v is None else f"{val_v:+.0%}",
            "배당률(FY)": "-" if yld_disp is None else f"{yld_disp:.2%}",
            "52주낙폭": "-" if dd_show is None else f"{dd_show:.1%}",
            "매도·상태 요약": sig_txt,
            "액션 판정": action,
            "평가금액": _fmt_krw_man(h["value_krw"], mask) if has_amounts else "-",
            "전체비중": f"{w_tot:.1%}" if est_total_asset > 0 else "-",
            "_is_fin": j["is_fin"],
            "_value_krw": h["value_krw"],
            "_sell_flag": sell_flag,
            "_common": ctk,
            "_close": live_close or 0.0,
        })

        # 정밀 진단 카드용 1~9번 항목 요약
        tr_ref = (track == "total_return")
        v_map = [
            ("1배당", f"{j['v1']}(참고)" if tr_ref else j["v1"], idet.get("1", "")),
            ("2EPS", f"{j['v2']}(참고)" if tr_ref else j["v2"], idet.get("2", "")),
            ("3FCF", j["v3"], idet.get("3", "")),
            ("4스프레드", j["v4"], idet.get("4", "")),
            ("5환원", j["v5"], idet.get("5", "")),
            ("6자사주", j["v6"], idet.get("6", "")),
            ("7감당", j["v7"], idet.get("7", "")),
            ("8건전", j["v8"], idet.get("8", "")),
            ("9가격", j["v9"], idet.get("9", "")),
        ]
        cards.append({
            "name": disp_name, "ticker": tk, "track": {"dividend": "배당성장", "total_return": "총환원형", "cycle": "사이클"}.get(track, track),
            "sector": PATH_KO.get(str(j["path"]), str(j["path"])), "fs_tag": fs_tag,
            "action": action, "sell_flag": sell_flag, "weight": f"{w_tot:.1%}" if est_total_asset > 0 else "-",
            "dps_trend": dps_trend_str, "checks": v_map, "details": detail_lines,
        })

    hdf = pd.DataFrame(rows)
    div_rows = hdf[hdf["트랙"] != "사이클"]
    n_fin = int(div_rows["_is_fin"].sum())
    n_nonfin = int((~div_rows["_is_fin"]).sum())
    fin_val = whole_meta["fin_val"] if whole_meta.get("has_whole") else (float(hdf.loc[hdf["_is_fin"], "_value_krw"].sum()) if has_amounts else 0.0)
    fin_w_total = (fin_val / est_total_asset) if est_total_asset > 0 else None
    fin_w_kr = (fin_val / tot_val_kr) if tot_val_kr > 0 else None
    breached_single = whole_meta.get("breached_single", []) if whole_meta.get("has_whole") else [
        f"{rr['종목명']}({rr['_value_krw']/est_total_asset:.1%})"
        for _, rr in hdf.iterrows() if est_total_asset > 0 and rr["_value_krw"] / est_total_asset > single_cap + 1e-6
    ]

    meta = {
        "has_holdings": True,
        "has_amounts": has_amounts,
        "tot_val_kr": tot_val_kr,
        "est_total_asset": est_total_asset,
        "kr_ratio": (tot_val_kr / est_total_asset) if est_total_asset > 0 else kr_ratio,
        "n_total": len(hdf),
        "n_sell": int(hdf["_sell_flag"].sum()),
        "n_watch": int((hdf["액션 판정"] == "관찰 (보유 유지)").sum()),
        "n_fin": n_fin,
        "n_nonfin": n_nonfin,
        "fin_val": fin_val,
        "fin_w_total": fin_w_total,
        "fin_w_kr": fin_w_kr,
        "single_cap": single_cap,
        "fin_cap": fin_cap,
        "fin_breach": bool(fin_w_total is not None and fin_w_total >= fin_cap - 1e-6),
        "breached_single": breached_single,
    }
    return hdf, cards, meta

def build_rebalance_plan(hdf: pd.DataFrame, hmeta: dict, whole_meta: dict, top_pool: list[dict], pf: dict) -> tuple[pd.DataFrame, dict]:
    new_cash = float(pf.get("new_cash_krw") or 0)
    sell_proceeds = 0.0
    if pf.get("include_sell_proceeds") and not hdf.empty and hmeta.get("has_amounts"):
        sell_proceeds = float(hdf.loc[hdf["_sell_flag"], "_value_krw"].sum())
    deploy_budget = new_cash + sell_proceeds
    if deploy_budget <= 0 or not top_pool:
        return pd.DataFrame(), {"active": False, "deploy_budget": deploy_budget, "new_cash": new_cash, "sell_proceeds": sell_proceeds}

    mask = bool(SCREENER_CONFIG["MASK_AMOUNTS_IN_REPORT"])
    single_cap = float(pf.get("single_stock_cap", 0.15))
    fin_cap = float(pf.get("financial_cap", 0.35))
    kr_ratio = max(0.05, min(1.0, float(pf.get("kr_basket_ratio_of_total", 0.50))))
    max_picks = max(1, int(pf.get("max_new_picks", 4)))
    min_liq = float(pf.get("prefer_liquidity_eok", 10.0))

    cur_val_by_common = {}
    cur_fin_val = 0.0
    cur_kr_val = 0.0
    if not hdf.empty and hmeta.get("has_amounts"):
        for _, rr in hdf.iterrows():
            if pf.get("include_sell_proceeds") and rr["_sell_flag"]:
                continue
            v = float(rr["_value_krw"])
            cur_val_by_common[rr["_common"]] = cur_val_by_common.get(rr["_common"], 0.0) + v
            cur_kr_val += v
            if rr["_is_fin"]:
                cur_fin_val += v

    if whole_meta.get("has_whole") and whole_meta["tot_val"] > 0:
        post_asset_total = whole_meta["tot_val"] - sell_proceeds + deploy_budget
    elif hmeta.get("has_amounts"):
        post_asset_total = (cur_kr_val + deploy_budget) / kr_ratio
    else:
        post_asset_total = deploy_budget / kr_ratio

    elig = [x for x in top_pool if x["v9"] != "WARN" and x["turnover_60d_eok"] >= min_liq]
    if len(elig) < max_picks:
        elig = elig + [x for x in top_pool if x["v9"] != "WARN" and x not in elig]

    fin_room = max(0.0, fin_cap * post_asset_total - cur_fin_val)
    chosen = []
    for x in elig:
        if x["is_fin"] and fin_room <= 100_000:
            continue
        cur_s = cur_val_by_common.get(x["ticker"], 0.0)
        s_room = max(0.0, single_cap * post_asset_total - cur_s)
        if s_room <= 100_000:
            continue
        chosen.append((x, cur_s, s_room))
        if len(chosen) >= max_picks:
            break

    if not chosen:
        return pd.DataFrame(), {"active": True, "deploy_budget": deploy_budget, "allocated": 0.0, "leftover": deploy_budget, "note": "모든 상위 후보가 종목·금융 상한에 도달했습니다."}

    alloc = {x["ticker"]: 0.0 for x, _, _ in chosen}
    rem = deploy_budget
    rem_fin = fin_room
    for _ in range(5):
        active_list = []
        for x, cur_s, s_room in chosen:
            cap_left = s_room - alloc[x["ticker"]]
            if x["is_fin"]:
                cap_left = min(cap_left, rem_fin)
            if cap_left > 10_000:
                active_list.append((x, cap_left))
        if not active_list or rem <= 10_000:
            break
        share = rem / len(active_list)
        spent = 0.0
        for x, cap_left in active_list:
            add = min(share, cap_left)
            if x["is_fin"]:
                add = min(add, rem_fin)
                rem_fin -= add
            alloc[x["ticker"]] += add
            spent += add
        rem -= spent
        if spent <= 10_000:
            break

    plan_rows = []
    for x, cur_s, _ in chosen:
        amt = alloc[x["ticker"]]
        if amt <= 0:
            continue
        px = float(x["close"] or 0)
        qty = int(amt // px) if px > 0 else 0
        post_w = (cur_s + amt) / post_asset_total if post_asset_total > 0 else 0.0
        plan_rows.append({
            "우선순위": x.get("overall_rank", "-"),
            "구분": x.get("hold_tag", "신규"),
            "종목명": x["name"],
            "티커": x["ticker"],
            "업종": "금융" if x["is_fin"] else "비금융",
            "가격(9번)": x["v9"],
            "핵심 강점": x.get("strength", "-"),
            "현재가": f"{px:,.0f}원" if px > 0 else "-",
            "권장 매수액": "비공개" if mask else f"{amt:,.0f}원",
            "예상 주수": "-" if (mask or qty <= 0) else f"{qty:,}주",
            "매수후 전체비중": f"{post_w:.1%}",
        })

    plan_df = pd.DataFrame(plan_rows)
    alloc_txt = f"{len(plan_df)}종목 배분 완료" if mask else f"신규 매수 {len(plan_df)}종목에 {deploy_budget - rem:,.0f}원 배분 (잔여 {rem:,.0f}원)"
    return plan_df, {
        "active": True,
        "deploy_budget": deploy_budget,
        "new_cash": new_cash,
        "sell_proceeds": sell_proceeds,
        "allocated": deploy_budget - rem,
        "leftover": rem,
        "post_asset_total": post_asset_total,
        "fin_room_init": fin_room,
        "note": alloc_txt,
    }

BASKET_DF, CAP_DF, WHOLE_META = analyze_whole_portfolio(MY_PF, JUDGED_MAP)
HOLDINGS_DF, HOLDING_CARDS, HOLD_META = evaluate_my_holdings(MY_HOLDINGS, JUDGED_MAP, PREF_PX_MAP, MY_PF, WHOLE_META)
PLAN_DF, PLAN_META = build_rebalance_plan(HOLDINGS_DF, HOLD_META, WHOLE_META, TOP_POOL, MY_PF)

if WHOLE_META.get("has_whole"):
    print("■ [전체 포트폴리오 바구니 비중] (계좌명·소유자명 비노출 집계)")
    display(BASKET_DF)
    print("\n■ [3대 상한선 점검] (운용규칙 3절)")
    display(CAP_DF)

if HOLD_META["has_holdings"]:
    print("\n■ [내 보유 한국주 정기 검진] (운용규칙 4절 A + 상한선 반영)")
    disp_cols = [c for c in HOLDINGS_DF.columns if not c.startswith("_")]
    display(HOLDINGS_DF[disp_cols])
else:
    print("[안내] 보유 종목이 연결되지 않아 전 시장 스크리닝 모드로 진행했습니다.")

if PLAN_META.get("active") and not PLAN_DF.empty:
    print(f"\n■ [4~5월 신규·배당 자금 매수 배분 가이드] ({PLAN_META['note']})")
    display(PLAN_DF)

print("\n■ [비금융 원칙 부합 TOP 10] (핵심 강점 포함)")
display(TOP_NONFIN_VIEW.head(10))
print("\n■ [금융 원칙 부합 TOP 10]")
display(TOP_FIN_VIEW.head(10))


In [ ]:
# ==============================================================================
# 8. 기관 리서치급 시각화 · 가독성 강화 A4 PDF & HTML 리포트 생성 · Drive 저장
# ==============================================================================
def export_html_to_pdf(html_str: str, pdf_path: str, chart_b64_list: list[tuple[str, str]] | None = None) -> bool:
    """HTML -> 고화질 A4 PDF 변환 (WeasyPrint 우선, 실패 시 Matplotlib 멀티페이지 PDF 폴백)."""
    try:
        from weasyprint import HTML as WP_HTML
        WP_HTML(string=html_str).write_pdf(pdf_path)
        return True
    except Exception as e:
        print(f"  [참고] WeasyPrint 변환 건너뜀({type(e).__name__}) -> Matplotlib PDF 폴백 생성 중...")
    try:
        from matplotlib.backends.backend_pdf import PdfPages
        import matplotlib.image as mpimg
        with PdfPages(pdf_path) as pdf:
            for title, b64 in (chart_b64_list or []):
                img = mpimg.imread(io.BytesIO(base64.b64decode(b64)), format="png")
                fig_p = plt.figure(figsize=(11.69, 8.27))
                plt.imshow(img)
                plt.axis("off")
                plt.title(title, fontsize=13, fontweight="bold", pad=12)
                plt.tight_layout()
                pdf.savefig(fig_p, dpi=150)
                plt.close(fig_p)
        return True
    except Exception as e2:
        print(f"  [주의] PDF 생성 실패: {e2}")
        return False

def place_scatter_labels(ax, pts: list[tuple[float, float, str, bool]], xlim: tuple[float, float], ylim: tuple[float, float]):
    """겹침 없는 라벨 배치 알고리즘 (충돌 회피 오프셋 탐색 + 얇은 지시선 연결)."""
    xr = max(1.0, xlim[1] - xlim[0])
    yr = max(1.0, ylim[1] - ylim[0])
    placed_boxes = []  # (x0, y0, x1, y1) in normalized axis coords
    # 후보 오프셋 (정규화 좌표 기준: 우상, 우하, 좌상, 좌하, 상, 하, 우측 멀리, 좌측 멀리 등)
    offsets = [
        (0.020, 0.032), (0.020, -0.048), (-0.115, 0.032), (-0.115, -0.048),
        (0.000, 0.068), (0.000, -0.075), (0.055, 0.065), (0.055, -0.075),
        (-0.155, 0.065), (-0.155, -0.075), (0.095, 0.005), (-0.185, 0.005),
        (0.020, 0.110), (0.020, -0.115), (-0.130, 0.110), (-0.130, -0.115),
    ]
    for x, y, text, is_hl in pts:
        nx = (x - xlim[0]) / xr
        ny = (y - ylim[0]) / yr
        w = 0.0225 * len(text) + 0.032
        h = 0.062
        best_pos = None
        best_pen = 1e9
        for dx, dy in offsets:
            tx = max(0.015, min(0.985 - w, nx + dx))
            ty = max(0.025, min(0.955 - h, ny + dy))
            box = (tx, ty, tx + w, ty + h)
            pen = 0.0
            for bx0, by0, bx1, by1 in placed_boxes:
                ox = max(0.0, min(box[2], bx1) - max(box[0], bx0))
                oy = max(0.0, min(box[3], by1) - max(box[1], by0))
                if ox > 0 and oy > 0:
                    pen += 100.0 * ox * oy + 5.0
            # 원래 점과의 거리 약하게 패널티
            pen += math.hypot(tx - nx, ty - ny) * 0.6
            if pen < best_pen:
                best_pen = pen
                best_pos = (tx, ty, box)
            if pen < 0.12:
                break
        tx, ty, box = best_pos
        placed_boxes.append(box)
        ax_x = xlim[0] + tx * xr
        ax_y = ylim[0] + (ty + h * 0.35) * yr
        ax.annotate(
            text, xy=(x, y), xytext=(ax_x, ax_y),
            fontsize=8.2 if is_hl else 7.8,
            fontweight="bold" if is_hl else "normal",
            color="#0F2922" if is_hl else "#334155",
            bbox=dict(boxstyle="round,pad=0.18", facecolor="#FEF3C7" if is_hl else "white",
                      edgecolor="#D97706" if is_hl else "#E2E8F0", alpha=0.92, lw=0.7),
            arrowprops=dict(arrowstyle="-", color="#94A3B8", lw=0.65, shrinkA=2, shrinkB=4),
            zorder=6 if is_hl else 4,
        )

KST = timezone(timedelta(hours=9))
as_of_str = datetime.now(KST).strftime("%Y-%m-%d")
run_stamp = datetime.now(KST).strftime("%Y-%m-%d_%H%M%S")

# ==============================================================================
# 차트 1: 3패널 기관 리서치 대시보드 (① 깔때기  ② 무겹침 사분면 맵  ③ 비금융 TOP 10 비교)
# ==============================================================================
fig = plt.figure(figsize=(15.4, 8.6), facecolor="white")
gs = fig.add_gridspec(2, 2, height_ratios=[1.12, 0.92], width_ratios=[0.85, 1.35], hspace=0.34, wspace=0.22)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

fig.suptitle("한국 배당성장주 통합 스크리닝 · 시각 분석 대시보드", fontsize=15.5, fontweight="bold", color="#0F2922", y=0.985)
fig.text(
    0.5, 0.946,
    f"기준일 {as_of_str}  |  사업연도 FY{FY}  |  대상 {', '.join(SCREENER_CONFIG['MARKETS'])} 시총 {SCREENER_CONFIG['MIN_MCAP_EOK']:,}억 원↑  |  유동성 배지 60일 {MY_PF.get('prefer_liquidity_eok', 10.0):.0f}억↑",
    ha="center", fontsize=9.4, color="#64748B",
)

# (1) 가로형 깔때기 바 차트 (여백 최소화, 직관적)
stages_h = ["최종 원칙 부합\n(비금융/금융)", "2차 정밀 판정", "1차 배당 통과", "탐색 모집단"]
counts_h = [len(TOP_DF), len(JUDGED), len(SURVIVORS), len(uni)]
cols_h = ["#0F766E", "#0284C7", "#38BDF8", "#CBD5E1"]
bars1 = ax1.barh(stages_h, counts_h, color=cols_h, height=0.56, edgecolor="white", lw=0.8)
max_c = max(counts_h + [1])
for idx_b, b in enumerate(bars1):
    w = b.get_width()
    lbl = f" {int(w)}개 (비금융 {len(TOP_NONFIN_DF)} · 금융 {len(TOP_FIN_DF)})" if idx_b == 0 else f" {int(w):,}개"
    ax1.text(w + max_c * 0.015, b.get_y() + b.get_height() / 2, lbl,
             va="center", ha="left", fontweight="bold", fontsize=9.2, color="#0F172A")
ax1.set_xlim(0, max_c * 1.38)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.grid(True, axis="x", color="#E2E8F0", ls="--", lw=0.7)
ax1.grid(False, axis="y")
ax1.set_title("① 스크리닝 단계별 생존 종목 수", fontsize=11.2, fontweight="bold", color="#0F2922", loc="left", pad=8)

# (2) 무겹침 사분면 맵 + Sweet Spot 하이라이트
if not TOP_DF.empty:
    xs_all = (TOP_DF[val_col].clip(-0.55, 1.15) * 100).tolist()
    ys_all = (TOP_DF["dps_g5"].clip(-0.05, 0.85) * 100).tolist()
    xmin = min(-55, min(xs_all) - 18)
    xmax = max(95, max(xs_all) + 26)
    ymin = -4
    ymax = max(55, max(ys_all) + 14)

    # Sweet Spot 음영 (밸류할인 > 0% & DPS 성장 > 10%)
    ax2.axvspan(0, xmax, ymin=(10 - ymin) / (ymax - ymin), ymax=1.0, color="#ECFDF5", alpha=0.65, zorder=0)
    ax2.text(xmax - 3, ymax - 3.5, "핵심 매력 구간\n(저평가 + 배당 고성장)", ha="right", va="top",
             fontsize=8.2, fontweight="bold", color="#047857", alpha=0.85)

    nonfin_p = TOP_DF[~TOP_DF["is_fin"]]
    fin_p = TOP_DF[TOP_DF["is_fin"]]
    if not nonfin_p.empty:
        ax2.scatter(
            nonfin_p[val_col].clip(-0.55, 1.15) * 100, nonfin_p["dps_g5"].clip(-0.05, 0.85) * 100,
            s=(nonfin_p["yield_fy"].fillna(0.02) * 2800).clip(65, 290),
            c="#059669", marker="o", alpha=0.85, edgecolors="#064E3B", lw=0.9, label="비금융 후보 (원 크기=배당률)", zorder=3,
        )
    if not fin_p.empty:
        ax2.scatter(
            fin_p[val_col].clip(-0.55, 1.15) * 100, fin_p["dps_g5"].clip(-0.05, 0.85) * 100,
            s=(fin_p["yield_fy"].fillna(0.02) * 2800).clip(65, 290),
            c="#2563EB", marker="s", alpha=0.85, edgecolors="#1E3A8A", lw=0.9, label="금융 후보", zorder=3,
        )
    held_p = TOP_DF[TOP_DF["ticker"].isin(HELD_COMMON_SET)]
    if not held_p.empty:
        ax2.scatter(
            held_p[val_col].clip(-0.55, 1.15) * 100, held_p["dps_g5"].clip(-0.05, 0.85) * 100,
            s=280, c="#F59E0B", marker="*", edgecolors="#78350F", lw=1.0, zorder=5, label="현재 보유중",
        )
    ax2.axvline(0, color="#94A3B8", ls="--", lw=0.9, zorder=1)
    ax2.axhline(10, color="#94A3B8", ls="--", lw=0.9, zorder=1)
    ax2.set_xlim(xmin, xmax)
    ax2.set_ylim(ymin, ymax)

    # 상위 14종목만 충돌 회피 스마트 라벨링 (보유/관심 우선 강조)
    pts_label = []
    for _, row in TOP_DF.head(14).iterrows():
        xv = float(max(-55, min(115, (row[val_col] or 0) * 100)))
        yv = float(max(-5, min(85, (row["dps_g5"] or 0) * 100)))
        is_hl = row["ticker"] in HELD_COMMON_SET or row["ticker"] in WATCH_COMMON_SET
        pts_label.append((xv, yv, row["name"], is_hl))
    place_scatter_labels(ax2, pts_label, (xmin, xmax), (ymin, ymax))
    ax2.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#E2E8F0", fontsize=8.3)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.grid(True, color="#E2E8F0", ls="--", lw=0.7)
ax2.set_title("② 원칙 부합 후보 사분면 (충돌 방지 스마트 라벨)", fontsize=11.2, fontweight="bold", color="#0F2922", loc="left", pad=8)
ax2.set_xlabel("밸류에이션 할인율 (+가 과거 10년 대비 저평가, %, ±115% 클립)", fontsize=9, color="#475569")
ax2.set_ylabel("정기 DPS 5년 성장률 (%)", fontsize=9, color="#475569")

# (3) 하단 전체 폭: 비금융 최우선 후보 TOP 10 핵심 지표 비교 (밸류할인 · DPS 성장률 · FY 배당률)
top_bar_df = TOP_NONFIN_DF.head(10).copy() if not TOP_NONFIN_DF.empty else TOP_DF.head(10).copy()
if not top_bar_df.empty:
    idx_x = np.arange(len(top_bar_df))
    w_b = 0.27
    v_vals = (top_bar_df[val_col].fillna(0).clip(-0.4, 1.0) * 100).values
    g_vals = (top_bar_df["dps_g5"].fillna(0).clip(0, 0.85) * 100).values
    y_vals = (top_bar_df["yield_fy"].fillna(0) * 100).values

    b_v = ax3.bar(idx_x - w_b, v_vals, width=w_b, color="#0F766E", label="밸류 할인율 (+저평가, %)", edgecolor="white")
    b_g = ax3.bar(idx_x, g_vals, width=w_b, color="#2563EB", label="DPS 5년 성장률 (%)", edgecolor="white")
    b_y = ax3.bar(idx_x + w_b, y_vals, width=w_b, color="#D97706", label="FY 배당수익률 (%)", edgecolor="white")

    for bars_grp in (b_v, b_g, b_y):
        for bb in bars_grp:
            hv = bb.get_height()
            va = "bottom" if hv >= 0 else "top"
            off = 1.5 if hv >= 0 else -1.5
            ax3.text(bb.get_x() + bb.get_width() / 2, hv + off, f"{hv:.0f}%" if abs(hv) >= 10 else f"{hv:.1f}%",
                     ha="center", va=va, fontsize=7.6, color="#334155", fontweight="bold")
    ax3.axhline(0, color="#64748B", lw=0.8)
    ax3.set_xticks(idx_x)
    x_labels = [f"{r['name']}\n({r['hold_tag']})" for _, r in top_bar_df.iterrows()]
    ax3.set_xticklabels(x_labels, fontsize=8.8, fontweight="bold", color="#0F172A")
    ax3.set_ylim(min(-18, min(v_vals) - 12), max(40, max(max(v_vals), max(g_vals)) + 16))
    ax3.legend(loc="upper right", ncol=3, frameon=True, facecolor="white", edgecolor="#E2E8F0", fontsize=8.6)
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
ax3.grid(True, axis="y", color="#E2E8F0", ls="--", lw=0.7)
ax3.grid(False, axis="x")
ax3.set_title("③ 비금융 최우선 후보 TOP 10 핵심 매력도 한눈에 비교 (밸류할인 · 배당성장 · 배당률)", fontsize=11.2, fontweight="bold", color="#0F2922", loc="left", pad=8)
ax3.set_ylabel("비율 (%)", fontsize=9, color="#475569")

fig.subplots_adjust(top=0.90, bottom=0.08, left=0.09, right=0.98)
buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=155, bbox_inches="tight", facecolor="white")
CHART_B64 = base64.b64encode(buf.getvalue()).decode()
plt.show()
plt.close(fig)

# ==============================================================================
# HTML / PDF 스타일 및 배지·숫자 강조 렌더러
# ==============================================================================
def scr_badge(val: str) -> str:
    s = str(val)
    if s in ("PASS", "통과", "양호", "정상", "정상 범위", "보유 유지 (추매 가능)", "신규") or s.startswith("원칙 부합"):
        return f'<span class="pill pill-green">{escape(s)}</span>'
    if s in ("보유중", "관심") or s.startswith("별도 트랙"):
        return f'<span class="pill pill-blue">{escape(s)}</span>'
    if s in ("WARN", "부분 부합", "판정 보류", "관찰 (보유 유지)", "상한 임박", "허용폭 미달") or s.startswith(("주의", "보유 유지 (금융상한", "보유 유지 (단일상한", "보유 유지 (가격부담")):
        return f'<span class="pill pill-amber">{escape(s)}</span>'
    if s in ("FAIL", "미부합", "매도 검토", "수집불가", "허용폭 초과") or s.startswith("상한 초과"):
        return f'<span class="pill pill-red">{escape(s)}</span>'
    return escape(s)

BADGE_COLS = {"구분", "종합", "품질종합", "가격(9번)", "유동성", "액션 판정", "보완필터", "판정", "상태"}

def _style_cell(col: str, val) -> str:
    s = str(val)
    if col in BADGE_COLS:
        return scr_badge(s)
    if col == "티커":
        return f'<span class="mono-muted">{escape(s)}</span>'
    if col == "순위":
        return f'<span class="rank-num">{escape(s)}</span>'
    if col == "밸류할인" and s.endswith("%"):
        try:
            v = float(s.replace("%", "").replace("+", ""))
            if v >= 25:
                return f'<span class="num-good">{escape(s)}</span>'
            if v <= -20:
                return f'<span class="num-bad">{escape(s)}</span>'
        except Exception:
            pass
    if col == "배당률(FY)" and s.endswith("%"):
        try:
            v = float(s.replace("%", ""))
            if v >= 5.0:
                return f'<span class="num-accent">{escape(s)}</span>'
        except Exception:
            pass
    if col == "52주낙폭" and s.endswith("%"):
        try:
            v = float(s.replace("%", ""))
            if v <= -45:
                return f'<span class="num-bad">{escape(s)}</span>'
        except Exception:
            pass
    return escape(s)

def render_scr_table(df: pd.DataFrame) -> str:
    if df.empty:
        return "<p style='color:#64748B;font-size:10px;margin:4px 0;'>해당 종목 없음</p>"
    ths = "".join(f"<th>{escape(str(c))}</th>" for c in df.columns)
    trs = []
    for _, row in df.iterrows():
        tds = []
        for c in df.columns:
            cls = "col-left" if c in ("종목명", "바구니", "상한 항목", "매도·상태 요약", "핵심 강점", "체크포인트") else ("col-center" if c in ("순위", "구분", "티커", "업종", "트랙") else "")
            tds.append(f'<td class="{cls}">{_style_cell(c, row[c])}</td>')
        trs.append("<tr>" + "".join(tds) + "</tr>")
    return f'<table class="tbl"><thead><tr>{ths}</tr></thead><tbody>{"".join(trs)}</tbody></table>'

def _nanum_font_css() -> str:
    parts = []
    for weight, path in [(400, "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"),
                         (700, "/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf")]:
        if os.path.exists(path):
            parts.append(f"@font-face {{ font-family:'NanumPDF'; src:url('file://{path}'); font-weight:{weight}; }}")
    return "\n  ".join(parts)

top_nonfin_names = ", ".join(TOP_NONFIN_DF["name"].head(5).tolist()) if not TOP_NONFIN_DF.empty else "없음"
top_fin_names = ", ".join(TOP_FIN_DF["name"].head(5).tolist()) if not TOP_FIN_DF.empty else "없음"
liq_good_cnt = int(TOP_DF["liq_ok"].sum()) if not TOP_DF.empty else 0

# KPI 4개 (절대 중복 없도록 구성!)
kpi1_lab, kpi1_val, kpi1_sub = "시장 스캔 (모집단 → 1차)", f"{len(uni):,} → {len(SURVIVORS):,}", f"10년 배당이력 생존율 {len(SURVIVORS)/max(1,len(uni)):.1%}"
kpi2_lab, kpi2_val, kpi2_sub = "최종 원칙 부합 후보", f"{len(TOP_DF)}종목", f"비금융 {len(TOP_NONFIN_DF)} · 금융 {len(TOP_FIN_DF)} (유동성 양호 {liq_good_cnt})"

if HOLD_META["has_holdings"]:
    kpi3_lab = "내 보유 한국주 검진"
    kpi3_val = f"{HOLD_META['n_total']}종목 (매도 {HOLD_META['n_sell']})"
    kpi3_sub = f"관찰 {HOLD_META['n_watch']} · 비금융 {HOLD_META['n_nonfin']} / 금융 {HOLD_META['n_fin']}"
    kpi3_color = "#DC2626" if HOLD_META["n_sell"] > 0 else "#0F766E"
else:
    kpi3_lab = "비금융 최우선 1~2위"
    kpi3_val = " · ".join(TOP_NONFIN_DF["name"].head(2).tolist()) if not TOP_NONFIN_DF.empty else "-"
    kpi3_sub = f"금융 편중 완화 핵심 후보 {len(TOP_NONFIN_DF)}종목"
    kpi3_color = "#1D4ED8"

if WHOLE_META.get("has_whole"):
    kpi4_lab = "국내 금융 / AI 노출"
    kpi4_val = f"{WHOLE_META['fin_w']:.1%} / {WHOLE_META['ai_w']:.1%}"
    kpi4_sub = f"상한 {MY_PF['financial_cap']:.0%} / {MY_PF['ai_cap']:.0%}"
    kpi4_color = "#DC2626" if (WHOLE_META["fin_breach"] or WHOLE_META["ai_breach"]) else "#0F766E"
    bw = WHOLE_META["b_weights"]
    whole_li = (
        f"<li><b>전체 자산 배분 현황 (계좌명 비노출 합산):</b> "
        f"국내 배당성장(A) <b>{bw.get('A', 0):.1%}</b> · 반도체·AI(B) <b>{bw.get('B', 0):.1%}</b> · 미국 지수(C) <b>{bw.get('C', 0):.1%}</b> | "
        f"국내 금융 <b>{WHOLE_META['fin_w']:.1%}</b>(상한 {MY_PF['financial_cap']:.0%}) · AI 노출 <b>{WHOLE_META['ai_w']:.1%}</b>(상한 {MY_PF['ai_cap']:.0%})"
        + (f" · 단일 상한 초과: <b>{escape(', '.join(WHOLE_META['breached_single']))}</b>" if WHOLE_META["breached_single"] else "")
        + "</li>"
    )
else:
    kpi4_lab = "국고채 10년물 (요구수익률 COE)"
    kpi4_val = f"{CUR_RF:.2%} ({CUR_RF+0.05:.2%})"
    kpi4_sub = "4번 자본비용(WACC·COE) 기준 금리"
    kpi4_color = "#B45309"
    whole_li = ""

if HOLD_META["has_holdings"]:
    hold_li = (
        f"<li><b>보유 한국주 정기 검진 ({HOLD_META['n_total']}종목):</b> "
        f"매도 검토 <b>{HOLD_META['n_sell']}종목</b> · 관찰 <b>{HOLD_META['n_watch']}종목</b> · "
        f"배당 트랙 비금융 {HOLD_META['n_nonfin']} / 금융 {HOLD_META['n_fin']}. "
        + ("<b>국내 금융 상한 초과로 신규 한국주 자금은 비금융 후보 우선 투입</b>을 권장합니다." if HOLD_META["fin_breach"] else "국내 금융 상한 이내입니다.")
        + "</li>"
    )
else:
    diag_note = MY_PF.get("_sheet_diag", "")
    hold_li = (
        f"<li><b>개인 포트폴리오 연동 상태:</b> 현재 전 시장 스크리닝 모드입니다 ({escape(diag_note)}). "
        f"Colab Secrets에 <code>PORTFOLIO_SHEET_ID</code>를 켜고 실행하면 전체 바구니 비중·3대 상한선·보유주 검진이 1페이지에 자동 표시됩니다.</li>"
    )

# 1페이지 하단 스포트라이트 카드 (포트폴리오 미연동 시 빈 공간 대신 비금융 TOP 3 + 금융 TOP 2 하이라이트!)
spotlight_html = ""
if not WHOLE_META.get("has_whole") and not TOP_DF.empty:
    spot_rows = []
    for _, r in TOP_NONFIN_DF.head(3).iterrows():
        spot_rows.append(("비금융 TOP", "#0F766E", r))
    for _, r in TOP_FIN_DF.head(2).iterrows():
        spot_rows.append(("금융 TOP", "#1D4ED8", r))
    cards_s = []
    for tag, col_c, r in spot_rows:
        vv = "-" if pd.isna(r[val_col]) else f"{r[val_col]:+.0%}"
        gg = "-" if pd.isna(r["dps_g5"]) else f"{r['dps_g5']:+.1%}"
        yy = "-" if pd.isna(r["yield_fy"]) else f"{r['yield_fy']:.2%}"
        cards_s.append(f"""
        <div class="spot-card" style="border-top-color:{col_c};">
          <div class="spot-top"><span class="spot-tag" style="color:{col_c};">{tag} · 전체 {int(r['overall_rank'])}위</span><span class="mono-muted">{r['ticker']}</span></div>
          <div class="spot-name">{escape(r['name'])}</div>
          <div class="spot-metrics">
            <div><span>밸류할인</span><b>{vv}</b></div>
            <div><span>DPS 5년</span><b>{gg}</b></div>
            <div><span>FY 배당률</span><b>{yy}</b></div>
          </div>
          <div class="spot-foot">업종 {PATH_KO.get(r['path'], r['path'])} · 가격 {r['v9']} · 거래대금 {r['turnover_60d_eok']:,.0f}억</div>
        </div>""")
    spotlight_html = f"""
  <div class="card">
    <div class="sec-title"><span class="sec-num">TOP 5</span>이번 스크리닝 핵심 주목 종목 하이라이트 (비금융 3 · 금융 2)</div>
    <div class="spot-grid">{"".join(cards_s)}</div>
  </div>"""

# 섹션 번호 자동 채번
sec_idx = 0
def next_sec() -> str:
    global sec_idx
    sec_idx += 1
    return str(sec_idx).zfill(2)

whole_section_html = ""
if WHOLE_META.get("has_whole"):
    sn = next_sec()
    whole_section_html = f"""
  <div class="card allow-break">
    <div class="sec-title"><span class="sec-num">{sn}</span>전체 포트폴리오 바구니 비중 &amp; 3대 상한선 진단 (계좌·소유자명 비노출 합산)</div>
    <div style="display:grid;grid-template-columns:1.15fr 1fr;gap:12px;">
      <div>{render_scr_table(BASKET_DF)}</div>
      <div>{render_scr_table(CAP_DF)}</div>
    </div>
  </div>"""

holdings_section_html = ""
if HOLD_META["has_holdings"]:
    sn = next_sec()
    h_disp = HOLDINGS_DF[[c for c in HOLDINGS_DF.columns if not c.startswith("_")]]
    if not HOLD_META["has_amounts"]:
        h_disp = h_disp.drop(columns=["평가금액", "전체비중"], errors="ignore")
    def _mini_pill(v: str) -> str:
        s = str(v)
        if s.startswith("PASS") or s == "참고":
            return f'<span class="mpill mp-g">{escape(s)}</span>'
        if s.startswith("FAIL"):
            return f'<span class="mpill mp-r">{escape(s)}</span>'
        if s.startswith("WARN"):
            return f'<span class="mpill mp-a">{escape(s)}</span>'
        return f'<span class="mpill mp-n">{escape(s)}</span>'

    diag_cards = []
    for hc in HOLDING_CARDS:
        bdr = "#DC2626" if hc["sell_flag"] else ("#D97706" if ("관찰" in hc["action"] or "상한" in hc["action"] or "중단" in hc["action"]) else "#0F766E")
        pills_html = "".join(
            f'<div class="mitem"><span class="mlab">{escape(lbl)}</span>{_mini_pill(ver)}</div>'
            for lbl, ver, _ in hc["checks"]
        )
        det_html = "".join(f'<li>{escape(line)}</li>' for line in hc["details"])
        diag_cards.append(f"""
        <div class="diag-card" style="border-left-color:{bdr};">
          <div class="diag-head">
            <div><span class="diag-name">{escape(hc['name'])}</span> <span class="mono-muted">{hc['ticker']} · {hc['sector']} · {hc['track']} · {hc['fs_tag']} · 비중 {hc['weight']}</span></div>
            <div>{scr_badge(hc['action'])}</div>
          </div>
          <div class="mgrid">{pills_html}</div>
          <ul class="diag-ul">{det_html}</ul>
          <div class="diag-dps">최근 3년 보통주 정기 DPS: <b>{escape(hc['dps_trend'])}</b></div>
        </div>""")

    holdings_section_html = f"""
  <div class="card page-break allow-break">
    <div class="sec-title"><span class="sec-num">{sn}</span>내 보유 한국주 정기 검진 (운용규칙 4절 A + 상한선 반영 · {len(h_disp)}종목)</div>
    <div class="scroll">{render_scr_table(h_disp)}</div>
  </div>
  <div class="card allow-break">
    <div class="sec-title"><span class="sec-num">{sn}-상세</span>보유 종목별 1~9번 정밀 진단 · 매도/관찰 사유 해설</div>
    <div class="diag-grid">{"".join(diag_cards)}</div>
  </div>"""

plan_section_html = ""
if PLAN_META.get("active") and not PLAN_DF.empty:
    sn = next_sec()
    plan_section_html = f"""
  <div class="card allow-break">
    <div class="sec-title"><span class="sec-num">{sn}</span>4~5월 신규·배당 자금 매수 배분 가이드 ({escape(PLAN_META["note"])})</div>
    <p style="margin:0 0 7px;font-size:9.8px;color:#475569;">
      원칙 부합 + 가격 WARN 제외 + 60일 거래대금 {MY_PF.get("prefer_liquidity_eok", 10.0):.0f}억 이상 우선 선정 후, 단일 종목 {MY_PF.get("single_stock_cap", 0.15):.0%} · 국내 금융 {MY_PF.get("financial_cap", 0.35):.0%} 상한을 반영한 참고 배분표입니다.
    </p>
    <div class="scroll">{render_scr_table(PLAN_DF)}</div>
  </div>"""

sec_chart = next_sec()
sec_nonfin = next_sec()
sec_fin = next_sec()
sec_reserve = next_sec()

pdf_cols_cand = ["순위", "구분", "종목명", "업종", "가격(9번)", "밸류할인", "DPS 5년", "배당률(FY)", "총환원율(3년)", "스프레드", "52주낙폭", "거래대금(억)", "유동성", "핵심 강점"]
TOP_NONFIN_PDF = TOP_NONFIN_VIEW[[c for c in pdf_cols_cand if c in TOP_NONFIN_VIEW.columns]].head(15) if not TOP_NONFIN_VIEW.empty else pd.DataFrame()
TOP_FIN_PDF = TOP_FIN_VIEW[[c for c in pdf_cols_cand if c in TOP_FIN_VIEW.columns]].head(15) if not TOP_FIN_VIEW.empty else pd.DataFrame()

# 매수 최우선 비금융 TOP 3 강점·체크포인트 요약 카드
buy_cards_html = ""
if not TOP_NONFIN_DF.empty:
    bc_items = []
    for _, r in TOP_NONFIN_DF.head(3).iterrows():
        vv = "-" if pd.isna(r[val_col]) else f"{r[val_col]:+.0%}"
        gg = "-" if pd.isna(r["dps_g5"]) else f"{r['dps_g5']:+.1%}"
        yy = "-" if pd.isna(r["yield_fy"]) else f"{r['yield_fy']:.2%}"
        sp = "-" if pd.isna(r["spread_5y"]) else f"{r['spread_5y']:+.1%}"
        bc_items.append(f"""
        <div class="buy-card">
          <div class="buy-head">
            <div><span class="buy-rank">비금융 추천 {int(r['overall_rank'])}위</span> <b>{escape(r['name'])}</b> <span class="mono-muted">{r['ticker']}</span></div>
            <div>{scr_badge(r['hold_tag'])} {scr_badge(r['v9'])}</div>
          </div>
          <div class="buy-kpis">
            <div><span>밸류할인</span><b>{vv}</b></div>
            <div><span>DPS 5년</span><b>{gg}</b></div>
            <div><span>FY 배당률</span><b>{yy}</b></div>
            <div><span>스프레드</span><b>{sp}</b></div>
          </div>
          <div class="buy-str"><b>핵심 강점:</b> {escape(r.get('strength', '-'))}</div>
          <div class="buy-cau"><b>체크포인트:</b> {escape(r.get('caution', '-'))}</div>
        </div>""")
    buy_cards_html = f"""
  <div class="card">
    <div class="sec-title"><span class="sec-num">매수 포인트</span>비금융 최우선 후보 TOP 3 강점 · 체크포인트 요약</div>
    <div class="buy-grid">{"".join(bc_items)}</div>
  </div>"""
res_cols = ["종목명", "티커", "업종", "종합", "가격(9번)", "밸류할인", "DPS 5년", "배당률(FY)", "총환원율(3년)", "스프레드", "52주낙폭", "거래대금(억)", "유동성"]
RESERVE_PDF = RESERVE_VIEW[[c for c in res_cols if c in RESERVE_VIEW.columns]].head(20) if not RESERVE_VIEW.empty else pd.DataFrame()
rejected_cnt = int((ALL_DF["overall"] == "미부합").sum()) if not ALL_DF.empty else 0

html_report = f"""<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<title>한국 배당성장주 4~5월 리밸런싱·스크리닝 리포트 ({as_of_str})</title>
<style>
  {_nanum_font_css()}
  @page {{
    size: A4 landscape;
    margin: 10mm 11mm 12mm 11mm;
    @bottom-left {{ content: "한국 배당성장주 4~5월 리밸런싱 · 통합 스크리닝 리포트 ({as_of_str})"; font-family: 'NanumPDF', sans-serif; font-size: 7.5pt; color: #94a3b8; }}
    @bottom-right {{ content: "Page " counter(page) " / " counter(pages); font-family: 'NanumPDF', sans-serif; font-size: 7.5pt; color: #94a3b8; }}
  }}
  :root {{
    --forest:#0F2922; --teal:#134E4A; --accent:#0F766E; --gold:#D4A34A;
    --ink:#0F172A; --sub:#334155; --muted:#64748B; --bg:#F8FAFC; --card:#FFFFFF; --line:#E2E8F0;
  }}
  * {{ box-sizing:border-box; }}
  body {{
    margin:0; padding:12px; background:var(--bg); color:var(--ink);
    font-family:'NanumPDF','NanumGothic',sans-serif; font-size:11px; line-height:1.52;
    font-variant-numeric:tabular-nums;
  }}
  .wrap {{ max-width:1120px; margin:0 auto; }}
  .masthead {{
    background:linear-gradient(135deg, var(--forest) 0%, var(--teal) 100%);
    color:#fff; padding:18px 22px; border-radius:12px; margin-bottom:12px;
    border-bottom:4px solid var(--gold);
  }}
  .eyebrow {{ font-size:8.8px; letter-spacing:1.6px; color:#FDE68A; font-weight:700; text-transform:uppercase; margin-bottom:4px; }}
  .masthead h1 {{ margin:0 0 4px; font-size:19.5px; font-weight:700; letter-spacing:-0.3px; }}
  .masthead .sub {{ margin:0; font-size:10.5px; color:#CBD5E1; }}
  .meta-row {{ display:flex; flex-wrap:wrap; gap:6px; margin-top:9px; }}
  .chip {{ background:rgba(255,255,255,0.12); border:1px solid rgba(255,255,255,0.22); padding:2.5px 9px; border-radius:999px; font-size:9.5px; color:#F1F5F9; }}

  .kpis {{ display:grid; grid-template-columns:repeat(4, 1fr); gap:10px; margin-bottom:12px; break-inside:avoid; }}
  .kpi {{ background:var(--card); border:1px solid var(--line); border-top:3.5px solid var(--accent); border-radius:10px; padding:11px 13px; }}
  .kpi .lab {{ font-size:9.5px; color:var(--muted); font-weight:700; }}
  .kpi .val {{ font-size:17px; font-weight:700; margin-top:3px; color:var(--ink); }}
  .kpi .ksub {{ font-size:9px; color:var(--muted); margin-top:2px; }}

  .card {{ background:var(--card); border:1px solid var(--line); border-radius:10px; padding:13px 15px; margin-bottom:11px; break-inside:avoid; }}
  .card.allow-break {{ break-inside:auto; }}
  .sec-title {{ display:flex; align-items:center; gap:8px; margin:0 0 8px; font-size:12.8px; font-weight:700; color:var(--forest); }}
  .sec-num {{ display:inline-block; background:var(--forest); color:#fff; font-size:9px; padding:2px 7px; border-radius:5px; letter-spacing:0.3px; }}

  .exec-box {{ background:#F0FDF9; border:1px solid #A7F3D0; border-left:4px solid var(--accent); }}
  .glossary {{
    display:grid; grid-template-columns:repeat(4, 1fr); gap:8px;
    background:#F8FAFC; border:1px solid var(--line); border-radius:8px;
    padding:8px 12px; font-size:9.2px; color:var(--sub); margin-top:10px;
  }}
  .glossary b {{ color:var(--forest); }}

  .spot-grid {{ display:grid; grid-template-columns:repeat(5, 1fr); gap:9px; }}
  .spot-card {{ background:#FAFCFB; border:1px solid var(--line); border-top:3px solid var(--accent); border-radius:8px; padding:9px 10px; }}
  .spot-top {{ display:flex; justify-content:space-between; align-items:center; font-size:8.5px; font-weight:700; margin-bottom:3px; }}
  .spot-name {{ font-size:12.5px; font-weight:700; color:var(--ink); margin-bottom:6px; }}
  .spot-metrics {{ display:grid; grid-template-columns:repeat(3, 1fr); gap:4px; background:#fff; border:1px solid #EEF2F6; border-radius:6px; padding:5px 4px; text-align:center; }}
  .spot-metrics span {{ display:block; font-size:7.8px; color:var(--muted); }}
  .spot-metrics b {{ font-size:10px; color:var(--forest); }}
  
  .diag-grid {{ display:grid; grid-template-columns:repeat(2, 1fr); gap:9px; }}
  .diag-card {{ background:#FAFCFB; border:1px solid var(--line); border-left:4px solid var(--accent); border-radius:8px; padding:8px 10px; break-inside:avoid; }}
  .diag-head {{ display:flex; justify-content:space-between; align-items:center; gap:6px; margin-bottom:5px; }}
  .diag-name {{ font-size:11.5px; font-weight:700; color:var(--ink); }}
  .mgrid {{ display:grid; grid-template-columns:repeat(9, 1fr); gap:3px; margin-bottom:5px; }}
  .mitem {{ background:#fff; border:1px solid #EEF2F6; border-radius:4px; padding:2px 2px; text-align:center; }}
  .mlab {{ display:block; font-size:7px; color:var(--muted); }}
  .mpill {{ display:inline-block; font-size:7.2px; font-weight:700; padding:0.5px 3px; border-radius:3px; }}
  .mp-g {{ background:#DCFCE7; color:#166534; }}
  .mp-a {{ background:#FEF3C7; color:#92400E; }}
  .mp-r {{ background:#FEE2E2; color:#991B1B; }}
  .mp-n {{ background:#F1F5F9; color:#475569; }}
  .diag-ul {{ margin:0 0 4px; padding-left:14px; font-size:8.6px; color:var(--sub); line-height:1.45; }}
  .diag-dps {{ font-size:8px; color:var(--muted); border-top:1px dashed #E2E8F0; padding-top:3px; }}

  .buy-grid {{ display:grid; grid-template-columns:repeat(3, 1fr); gap:10px; }}
  .buy-card {{ background:#F0FDF9; border:1px solid #A7F3D0; border-top:3px solid var(--accent); border-radius:8px; padding:9px 11px; break-inside:avoid; }}
  .buy-head {{ display:flex; justify-content:space-between; align-items:center; gap:4px; margin-bottom:6px; font-size:11px; }}
  .buy-rank {{ display:inline-block; background:var(--forest); color:#fff; font-size:8px; padding:1.5px 5px; border-radius:4px; margin-right:3px; }}
  .buy-kpis {{ display:grid; grid-template-columns:repeat(4, 1fr); gap:4px; background:#fff; border:1px solid #D1FAE5; border-radius:6px; padding:5px 4px; text-align:center; margin-bottom:6px; }}
  .buy-kpis span {{ display:block; font-size:7.6px; color:var(--muted); }}
  .buy-kpis b {{ font-size:9.8px; color:var(--forest); }}
  .buy-str {{ font-size:8.8px; color:#065F46; margin-bottom:2px; }}
  .buy-cau {{ font-size:8.4px; color:#92400E; }}

  .spot-foot {{ font-size:8.2px; color:var(--muted); margin-top:5px; }}

  .scroll {{ overflow-x:auto; }}
  table.tbl {{ width:100%; border-collapse:collapse; font-size:9.8px; }}
  table.tbl thead {{ display:table-header-group; }}
  table.tbl tr {{ break-inside:avoid; }}
  table.tbl th {{
    background:#F1F5F9; color:#334155; font-weight:700; padding:6px 5px;
    border-top:1.5px solid var(--forest); border-bottom:1px solid #CBD5E1; text-align:right;
  }}
  table.tbl td {{ padding:5.5px 5px; border-bottom:1px solid #EEF2F6; text-align:right; color:var(--ink); }}
  table.tbl tbody tr:nth-child(even) {{ background:#F9FEFC; }}
  table.tbl th:first-child {{ text-align:center; }}
  table.tbl td.col-left {{ text-align:left; font-weight:700; color:var(--ink); }}
  table.tbl td.col-center {{ text-align:center; }}
  .mono-muted {{ color:#64748B; font-size:8.8px; font-weight:400; }}
  .rank-num {{ display:inline-block; min-width:18px; text-align:center; font-weight:700; color:var(--forest); }}
  .num-good {{ color:#047857; font-weight:700; }}
  .num-bad {{ color:#B91C1C; font-weight:700; }}
  .num-accent {{ color:#1D4ED8; font-weight:700; }}

  .pill {{ display:inline-block; padding:1.5px 6.5px; border-radius:999px; font-size:8.2px; font-weight:700; white-space:nowrap; }}
  .pill-green {{ background:#DCFCE7; color:#166534; }}
  .pill-blue {{ background:#DBEAFE; color:#1E40AF; }}
  .pill-amber {{ background:#FEF3C7; color:#92400E; }}
  .pill-red {{ background:#FEE2E2; color:#991B1B; }}
  img.chart {{ width:100%; height:auto; display:block; border-radius:6px; }}
  .page-break {{ page-break-before:always; break-before:page; }}
  .foot {{ font-size:8.8px; color:var(--muted); border-top:1px solid var(--line); padding-top:7px; margin-top:6px; }}
  @media print {{
    body {{ background:#fff; padding:0; font-size:8.8pt; }}
    .wrap {{ max-width:none; }}
    .scroll {{ overflow:visible; }}
    table.tbl {{ font-size:7.6pt; }}
    table.tbl th, table.tbl td {{ padding:4.2px 4px; }}
  }}
</style>
</head>
<body>
<div class="wrap">
  <!-- ======================== PAGE 1: EXECUTIVE SUMMARY ======================== -->
  <div class="masthead">
    <div class="eyebrow">KRX Annual Rebalancing &amp; Dividend Growth Investment Brief</div>
    <h1>한국 배당성장주 4~5월 리밸런싱 · 통합 스크리닝 리포트</h1>
    <p class="sub">전 시장 10년 배당·재무 공시 정량 검증, 개인 보유 포트폴리오 상한 진단, 비금융·금융 우선순위 추천</p>
    <div class="meta-row">
      <span class="chip">기준일 {as_of_str}</span>
      <span class="chip">사업연도 FY{FY}</span>
      <span class="chip">대상 {", ".join(SCREENER_CONFIG["MARKETS"])} 시총 {SCREENER_CONFIG["MIN_MCAP_EOK"]:,}억↑</span>
      <span class="chip">유동성 배지 60일 {MY_PF.get("prefer_liquidity_eok", 10.0):.0f}억↑</span>
      <span class="chip">연동 {escape(MY_PF["_source"])}</span>
    </div>
  </div>

  <div class="kpis">
    <div class="kpi"><div class="lab">{kpi1_lab}</div><div class="val">{kpi1_val}</div><div class="ksub">{kpi1_sub}</div></div>
    <div class="kpi" style="border-top-color:#059669"><div class="lab">{kpi2_lab}</div><div class="val" style="color:#059669">{kpi2_val}</div><div class="ksub">{kpi2_sub}</div></div>
    <div class="kpi" style="border-top-color:{kpi3_color}"><div class="lab">{kpi3_lab}</div><div class="val" style="color:{kpi3_color}">{kpi3_val}</div><div class="ksub">{kpi3_sub}</div></div>
    <div class="kpi" style="border-top-color:{kpi4_color}"><div class="lab">{kpi4_lab}</div><div class="val" style="color:{kpi4_color}">{kpi4_val}</div><div class="ksub">{kpi4_sub}</div></div>
  </div>

  <div class="card exec-box">
    <div class="sec-title" style="color:#065F46;"><span class="sec-num" style="background:#0F766E;">요약</span>30초 만에 읽는 핵심 결론</div>
    <ul style="margin:0;padding-left:18px;font-size:10.3px;color:#064E3B;line-height:1.68;">
      <li><b>시장 스캔 결과:</b> 전체 <b>{len(uni):,}개</b> 보통주 중 10년 배당 이력을 통과한 <b>{len(SURVIVORS):,}개</b>를 정밀 판정해, 최종 <b>원칙 부합 {len(TOP_DF)}종목</b>(비금융 {len(TOP_NONFIN_DF)} · 금융 {len(TOP_FIN_DF)})을 선별했습니다.</li>
      {whole_li}
      {hold_li}
      <li><b>비금융 최우선 후보 TOP 5:</b> <b>{escape(top_nonfin_names)}</b> &nbsp;|&nbsp; <b>금융 후보 TOP 5:</b> <b>{escape(top_fin_names)}</b></li>
    </ul>
    <div class="glossary">
      <div><b>밸류할인(+):</b> 과거 10년 평균 대비 저평가 폭</div>
      <div><b>DPS 5년:</b> 최근 5년 연평균 배당 성장률</div>
      <div><b>스프레드:</b> 자본비용(WACC·COE) 초과 수익률</div>
      <div><b>유동성 양호:</b> 60일 일평균 거래대금 {MY_PF.get("prefer_liquidity_eok", 10.0):.0f}억↑</div>
    </div>
  </div>

  {whole_section_html}
  {spotlight_html}

  <!-- ======================== PORTFOLIO AUDIT & PLAN (IF CONNECTED) ======================== -->
  {holdings_section_html}
  {plan_section_html}

  <!-- ======================== VISUAL DASHBOARD ======================== -->
  <div class="card page-break">
    <div class="sec-title"><span class="sec-num">{sec_chart}</span>시각 분석 대시보드 (깔때기 · 무겹침 사분면 맵 · 비금융 TOP 10 매력도)</div>
    <img class="chart" src="data:image/png;base64,{CHART_B64}" alt="스크리닝 대시보드">
  </div>

  <!-- ======================== ACTIONABLE TOP CANDIDATES ======================== -->
  <div class="page-break"></div>
  {buy_cards_html}
  <div class="card allow-break">
    <div class="sec-title"><span class="sec-num">{sec_nonfin}</span>비금융 원칙 부합 추천 후보 TOP {len(TOP_NONFIN_PDF)} (금융 편중 완화 최우선)</div>
    <div class="scroll">{render_scr_table(TOP_NONFIN_PDF)}</div>
  </div>

  <div class="card allow-break">
    <div class="sec-title"><span class="sec-num">{sec_fin}</span>금융 원칙 부합 추천 후보 TOP {len(TOP_FIN_PDF)} (금융 상한 여유 시 참고 · *규제자본비율 별도 확인)</div>
    <div class="scroll">{render_scr_table(TOP_FIN_PDF)}</div>
  </div>

  <!-- ======================== RESERVE WATCHLIST (CONCISE!) ======================== -->
  <div class="card page-break allow-break">
    <div class="sec-title"><span class="sec-num">{sec_reserve}</span>예비 승격 후보: 「부분 부합 · 판정 보류」 워치리스트 ({len(RESERVE_PDF)}종목)</div>
    <p style="margin:0 0 7px;font-size:9.6px;color:#475569;">
      1~8번 <b>FAIL은 0건</b>이지만 WARN이 3~4건이라 아쉽게 원칙 부합에서 빠진 예비 후보입니다. (탈락한 미부합 {rejected_cnt}종목 포함 전체 {len(ALL_VIEW)}종목 원본 데이터는 함께 저장된 CSV에서 확인하실 수 있습니다.)
    </p>
    <div class="scroll">{render_scr_table(RESERVE_PDF)}</div>
  </div>

  <div class="foot">
    ※ <b>개인정보 보호:</b> 계좌명·소유자명은 리포트에 포함되지 않으며, 모든 포트폴리오 데이터는 본인 비공개 Google 계정 권한으로만 처리됩니다. &nbsp;|&nbsp;
    *금융 업종(은행·보험·증권·금융지주)의 8번 규제자본비율(CET1·K-ICS·NCR)은 사업보고서 본문 공시를 별도 확인하세요. 본 리포트는 정량 연구·정리 참고 자료이며 투자 권유가 아닙니다.
  </div>
</div>
</body>
</html>"""

report_html = f"screener_report_{run_stamp}.html"
report_pdf = f"screener_report_{run_stamp}.pdf"
csv_file = f"screener_candidates_{run_stamp}.csv"
plan_csv_file = f"rebalance_plan_{run_stamp}.csv"

Path(report_html).write_text(html_report, encoding="utf-8")
pdf_ok = export_html_to_pdf(html_report, report_pdf, [("한국 배당성장주 리밸런싱·스크리닝 대시보드", CHART_B64)])
ALL_VIEW.to_csv(csv_file, index=False, encoding="utf-8-sig")
if PLAN_META.get("active") and not PLAN_DF.empty:
    PLAN_DF.to_csv(plan_csv_file, index=False, encoding="utf-8-sig")

print(f"[리포트 생성 완료] PDF={'성공 (' + str(round(Path(report_pdf).stat().st_size/1024, 1)) + ' KB)' if pdf_ok and Path(report_pdf).exists() else '실패'} | HTML={Path(report_html).stat().st_size/1024:,.1f} KB · CSV={csv_file}")
if DRIVE_REPORT_DIR is not None:
    try:
        if pdf_ok and Path(report_pdf).exists():
            (DRIVE_REPORT_DIR / report_pdf).write_bytes(Path(report_pdf).read_bytes())
        (DRIVE_REPORT_DIR / report_html).write_text(html_report, encoding="utf-8")
        ALL_VIEW.to_csv(DRIVE_REPORT_DIR / csv_file, index=False, encoding="utf-8-sig")
        if PLAN_META.get("active") and not PLAN_DF.empty:
            PLAN_DF.to_csv(DRIVE_REPORT_DIR / plan_csv_file, index=False, encoding="utf-8-sig")
        print(f"[Google Drive 자동 저장 완료] {DRIVE_REPORT_DIR / report_pdf}")
        print("  -> 폰 Google Drive 앱 -> Colab Notebooks/dividend_screener/reports/ 에서 PDF 바로 열기!")
    except Exception as e:
        print(f"[주의] Drive 리포트 저장 실패({type(e).__name__}): {e}")

# 리포트는 Google Drive(Colab Notebooks/dividend_screener/reports/)에만 자동 저장합니다 (브라우저 자동 다운로드 생략).
